# Microsoft Fabric Lakehouse Discovery, Optimization & Implementation Planning

This reusable notebook provides a read-only-first framework for assessing and improving Microsoft Fabric Lakehouses. It supports:

1. discovering Lakehouses, Delta tables, and OneLake shortcuts;
2. profiling active Delta files, table history, properties, and physical layout;
3. creating evidence-based recommendations and implementation plans;
4. generating reviewed candidate SQL/Python commands, prerequisites, validation tests, acceptance criteria, and rollback guidance;
5. optionally executing approved maintenance with exact-table approval controls.

`run_mode = "audit_only"` is the safe default. It does not modify Lakehouse tables or persist the audit log.


## Prerequisites

### Environment and permissions
- A Microsoft Fabric workspace on an active Fabric capacity with Spark/Notebook execution enabled.
- Runtime 2.0 or later is recommended. Liquid Clustering, NEE, and some maintenance behavior differ by runtime. [Fabric Spark runtimes](https://learn.microsoft.com/fabric/data-engineering/runtime)
- Workspace role with enough permission to read workspace items and run notebooks. Lakehouse discovery uses the Fabric REST API, so the executing identity needs access to the relevant items. [Workspace roles](https://learn.microsoft.com/fabric/fundamentals/roles-workspaces)
- Read access to the targeted Lakehouse tables, OneLake paths, Delta transaction logs, schemas, shortcut metadata, and table history.
- To use mutating modes, the identity must also have write permissions on the native target tables. Shortcut targets require maintenance by the source owner rather than from the consumer shortcut.

### Notebook and Spark setup
- Use the Microsoft Fabric Runtime PySpark kernel and `notebookutils`.
- The notebook obtains the current workspace and default Lakehouse context at runtime; an attached default Lakehouse is recommended but not required for workspace-wide discovery.
- Allow outbound access to `https://api.fabric.microsoft.com` for notebook-scoped REST discovery calls. [Notebook public APIs](https://learn.microsoft.com/fabric/data-engineering/notebook-public-api)
- Parameter overrides require the notebook `parameters` cell to remain tagged and use the supported Job Scheduler parameter shape.
- Sufficient Spark capacity is required for workspace-wide table profiling. Large estates may need longer execution windows and tuned `max_discovery_workers` / `max_parallel_tables` values.

### Data and workload preparation
- Tables must be readable Delta tables for full profiling. Non-Delta or inaccessible tables are reported and skipped.
- For shortcut-based tables, confirm that the executing identity can resolve both consumer metadata and, where possible, the source target. Use source-owner verification when direct target access is unavailable. [OneLake shortcuts](https://learn.microsoft.com/fabric/onelake/onelake-shortcuts)
- Confirm that representative query, write, mutation, and consumer workloads are available if recommendations must be benchmarked.
- Capture a baseline for row correctness/checksums, file count and size, bytes read, duration, and capacity usage before approved changes.

### Operational safeguards
- Keep `run_mode = "audit_only"` for the first run.
- Leave `require_explicit_plan_approval = True` and provide exact `lakehouse.schema.table` values in `approved_plan_table_fqns_json` before any mutating run.
- Keep `skip_maintenance_on_shortcuts = True`; apply shortcut recommendations only at the source owner.
- Keep `vacuum_retain_hours >= 168` unless an explicitly approved disposable test requires a shorter interval. [VACUUM Delta tables](https://learn.microsoft.com/fabric/data-engineering/delta-lake-vacuum)
- Review `data_lifecycle_purge_rules_json`; leave it empty unless row deletion is explicitly authorized.
- Keep `apply_recommended_table_props = False` until property changes have owner approval and workload evidence.
- Preserve a clone, Delta-version recovery path, downstream consumer inventory, and maintenance window for any approved mutation.

## Source and validation basis

This notebook uses public Microsoft Fabric/Delta capabilities described in Microsoft Learn:

- [Delta table optimization and V-Order](https://learn.microsoft.com/fabric/data-engineering/delta-optimization-and-v-order)
- [Table compaction and OPTIMIZE](https://learn.microsoft.com/fabric/data-engineering/table-compaction)
- [Tune Delta file size](https://learn.microsoft.com/fabric/data-engineering/tune-file-size)
- [Liquid Clustering](https://learn.microsoft.com/fabric/data-engineering/liquid-clustering)
- [File skipping](https://learn.microsoft.com/fabric/data-engineering/delta-lake-file-skipping)
- [Deletion vectors](https://learn.microsoft.com/fabric/data-engineering/deletion-vectors)
- [Native Execution Engine](https://learn.microsoft.com/fabric/data-engineering/native-execution-engine-overview)
- [OneLake shortcuts](https://learn.microsoft.com/fabric/onelake/onelake-shortcuts)
- [Spark performance best practices](https://learn.microsoft.com/fabric/data-engineering/spark-best-practices-performance-optimization)

## Important qualification

The notebook contains **custom assessment heuristics**: thresholds, health scores, inferred table layers, candidate clustering columns, and implementation-plan classifications. These are not Microsoft product guarantees or universal recommendations. Validate them with representative workloads before execution.

All workspace, Lakehouse, shortcut, table, and run-output data are discovered at runtime in the active environment. Review generated commands, target ownership, permissions, retention, concurrency, rollback, and downstream dependencies before implementation.


## Default Lakehouse configuration

A default Lakehouse is **not required for the safe `audit_only` mode**. The notebook discovers Lakehouse roots explicitly and reads them through OneLake paths.

- If you want persistent catalog tables or catalog-based commands such as `ANALYZE TABLE`, attach an existing Lakehouse from the Fabric notebook Lakehouse pane before running the mutating modes.
- Workspace-wide discovery is still not limited to that attached Lakehouse; the notebook uses explicit ABFSS paths.
- Keep `run_mode = "audit_only"` for the first run. It performs no table mutation and does not require persistence.
- If running cells individually, execute **Step 1: Runtime Parameters** before executing later code cells.
- Public reference: [Author and execute notebooks](https://learn.microsoft.com/fabric/data-engineering/author-execute-notebook)


---
# Executable Engine: Automated Lakehouse Discovery, Fine-Tuning & Maintenance
### How to Run This Notebook
1. **Step 1 — Discovery & Health Assessment (Safe Dry-Run Default):**
   * Leave `run_mode = "audit_only"` in **Cell 1 (Parameters)** below and run all cells.
   * The engine will scan every Delta table in your Lakehouse, evaluate all health heuristics, display prioritized action tables, and optionally log the baseline snapshot—**without modifying a single file or table property**.
2. **Step 2 — Automated Table Optimization & Storage Cleanup:**
   * Change `run_mode` to:
     * `"optimize_only"` — Runs intelligent `OPTIMIZE` (with `Fast Optimize`, `Adaptive Target File Size`, or scoped date predicates) only on tables that need compaction or DV purging.
     * `"vacuum_only"` — Runs safe `VACUUM RETAIN 168 HOURS` on tables with unreferenced orphan files.
     * `"full_maintenance"` — Executes optional **Data Lifecycle Purge** $\rightarrow$ **Table Property Fine-Tuning** $\rightarrow$ **`OPTIMIZE`** $\rightarrow$ **`VACUUM`** $\rightarrow$ **`ANALYZE TABLE` Statistics Refresh**, using parallel `FAIR` scheduler pools and writing a single batched audit record at completion.


## Shareability and provenance

The executable cells use public Fabric and Delta capabilities plus custom assessment logic:

- **Public capability:** `DESCRIBE DETAIL`, `DESCRIBE HISTORY`, `OPTIMIZE`, `VACUUM`, Delta statistics, table properties, file skipping, Liquid Clustering, V-Order, deletion vectors, OneLake shortcuts, NotebookUtils, and Spark monitoring.
- **Custom assessment logic:** file-count/size thresholds, score bands, table-layer inference, candidate-column inference, priority ordering, risk/owner/effort labels, and generated implementation plans.
- **Evidence-dependent:** V-Order, Native Execution Engine, Optimize Write, Auto Compaction, Liquid Clustering, CBO, and extended statistics require representative workload testing.
- **Environment-specific:** generated findings, IDs, table names, shortcut targets, and measured audit outputs must be reviewed before sharing outside the active environment.

No confidential training-deck references, internal performance multipliers, or internal capacity/startup/commercial tables are included in this notebook.


---
### Step 1: Runtime Parameters & Configuration Control Panel
**What this cell does:**
* Defines all user-configurable knobs for the notebook in one place.
* Tagged with the Fabric `parameters` tag so a Fabric Data Pipeline or notebook job can override supported values at runtime. [Notebook parameter cells](https://learn.microsoft.com/fabric/data-engineering/author-execute-notebook) · [Run on-demand item job](https://learn.microsoft.com/rest/api/fabric/core/job-scheduler/run-on-demand-item-job)
* **Safe by default:** `run_mode = "audit_only"` is a framework safety policy that performs a read-only assessment without modifying table data.

#### Public documentation map for these controls

| Control group | Public Microsoft Learn reference | Public guidance versus framework behavior |
|---|---|---|
| Notebook execution modes and parameter overrides | [Author and execute notebooks](https://learn.microsoft.com/fabric/data-engineering/author-execute-notebook) · [Run on-demand item job](https://learn.microsoft.com/rest/api/fabric/core/job-scheduler/run-on-demand-item-job) | Parameter cells and Jobs API are public. The `audit_only`, `optimize_only`, `vacuum_only`, and `full_maintenance` names are framework modes. |
| Workspace, Lakehouse, schema, and table discovery | [Search and explore OneLake with Fabric REST APIs](https://learn.microsoft.com/rest/api/fabric/articles/onelakecatalog/overview) · [NotebookUtils runtime](https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-runtime) | Discovery through Fabric APIs and runtime context is public. Include/exclude lists and scan scope are framework controls. |
| Delta `OPTIMIZE`, Auto Compaction, Fast Optimize, Optimize Write, and file-level targets | [Tune Delta file size](https://learn.microsoft.com/fabric/data-engineering/tune-file-size) · [Compact Delta tables](https://learn.microsoft.com/fabric/data-engineering/table-compaction) | The capabilities and configurations are public. Audit cutoffs such as 50 files, 64 MB, 128 MB, and 5 GB are configurable framework heuristics unless explicitly stated as a Microsoft default. |
| Adaptive target file size | [Tune Delta file size](https://learn.microsoft.com/fabric/data-engineering/tune-file-size#adaptive-target-file-size) | Adaptive target sizing is public and enabled by default in Runtime 2.0+. The notebook does not prescribe a custom target curve. |
| Deletion vectors and mutation layout | [Deletion vectors](https://learn.microsoft.com/fabric/data-engineering/deletion-vectors) | Deletion vectors are public. Whether a table should enable and benchmark them is workload-dependent framework guidance. |
| V-Order | [Delta optimization and V-Order](https://learn.microsoft.com/fabric/data-engineering/delta-optimization-and-v-order) | V-Order controls are public. The notebook’s consumer-specific test requirement is a safety heuristic, not a mandatory Microsoft policy. |
| Native Execution Engine | [Native Execution Engine](https://learn.microsoft.com/fabric/data-engineering/native-execution-engine-overview) | The feature and enablement controls are public. Keeping it opt-in reflects benchmark evidence; it is not a universal performance promise. |
| File skipping and statistics columns | [Delta file skipping](https://learn.microsoft.com/fabric/data-engineering/delta-lake-file-skipping) | File-level statistics, indexed-column behavior, and explicit statistics-column selection are public. Selecting inferred columns is framework logic and requires query evidence. |
| Partitioning and Liquid Clustering | [Delta table partitioning](https://learn.microsoft.com/fabric/data-engineering/delta-lake-partitioning) · [Liquid Clustering](https://learn.microsoft.com/fabric/data-engineering/liquid-clustering) | Public guidance supports workload-based key selection and warns against small/high-cardinality partitions. The 1 TB/10,000-partition screening values are framework heuristics. |
| Table statistics and CBO | [Automated Table Statistics](https://learn.microsoft.com/fabric/data-engineering/automated-table-statistics) | Statistics capabilities are public. When to force `ANALYZE TABLE` and whether to retain extended statistics are workload-dependent framework decisions. |
| `VACUUM`, retention, and `OPTIMIZE` sequencing | [VACUUM Delta tables](https://learn.microsoft.com/fabric/data-engineering/delta-lake-vacuum) · [Cross-workload maintenance](https://learn.microsoft.com/fabric/fundamentals/table-maintenance-optimization) | Seven-day/168-hour default retention is public. The notebook’s hard safety lock and cadence are framework safeguards. |
| OneLake shortcuts and source ownership | [OneLake shortcuts](https://learn.microsoft.com/fabric/onelake/onelake-shortcuts) · [List Shortcuts REST API](https://learn.microsoft.com/rest/api/fabric/core/onelake-shortcuts/list-shortcuts) | Shortcut target/path discovery is public. Blocking consumer-side `OPTIMIZE`/`VACUUM` is a framework single-writer safeguard. |
| Audit persistence, implementation plans, approval lists, scores, and ownership | No single Microsoft product requirement applies. | These controls are custom framework governance and assessment logic. They use public Delta tables, Notebook, REST, and Spark capabilities but are not Microsoft defaults. |

> **Interpretation rule:** A documented capability is public. An exact threshold, schedule, score, name-based classification, or approval policy in this notebook is a configurable heuristic unless the reference table above or its linked documentation explicitly documents that value.


In [ ]:
# ==============================================================================
# STEP 1: RUNTIME PARAMETERS (Fabric Data Pipeline Parameter-Cell Compatible)
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. EXECUTION MODE
# ------------------------------------------------------------------------------
# - "audit_only"       : 100% Read-only discovery, health scoring & DDL plan (SAFE DEFAULT)
# - "optimize_only"    : Runs intelligent OPTIMIZE on tables flagged as needing file compaction
# - "vacuum_only"      : Runs safe VACUUM (RETAIN >= 168 HOURS) on tables with unreferenced files
# - "full_maintenance" : Runs Lifecycle Purge -> Table Properties -> OPTIMIZE -> VACUUM -> ANALYZE
run_mode = "audit_only"

# ------------------------------------------------------------------------------
# 2. WORKSPACE & LAKEHOUSE DISCOVERY SCOPE
# ------------------------------------------------------------------------------
# - "all_workspace_lakehouses" : Automatically discovers & scans ALL Lakehouses in the workspace!
# - "current_lakehouse"        : Scans only the pinned Default Lakehouse (or target_lakehouse_name)
scan_scope = "all_workspace_lakehouses"

target_lakehouse_name = ""          # Optional comma-separated Lakehouse names to scan ("" = ALL Lakehouses)
lakehouse_exclude_list = ""         # Optional comma-separated Lakehouse names to skip
target_schemas = ""                 # Optional comma-separated schemas to scan ("" = all schemas)
table_include_list = ""             # Optional comma-separated tables to include ("" = all tables)
table_exclude_list = "dbo.lakehouse_optimization_audit_log,dbo.lakehouse_file_distribution_audit,lakehouse_optimization_audit_log"

# ------------------------------------------------------------------------------
# 3. BASELINE SESSION & TABLE PROPERTY SWITCHES
# References: https://learn.microsoft.com/fabric/data-engineering/tune-file-size
#             https://learn.microsoft.com/fabric/data-engineering/table-compaction
#             https://learn.microsoft.com/fabric/data-engineering/delta-optimization-and-v-order
#             https://learn.microsoft.com/fabric/data-engineering/native-execution-engine-overview
# ------------------------------------------------------------------------------
apply_golden_session_configs = True   # Enables safe Runtime 2.0, AQE, adaptive sizing, fast optimize, CBO controls
apply_nee_session_config = False      # NEE requires representative A/B evidence; do not force it for small workloads
apply_recommended_table_props = False # When True (and not audit_only), stamps DVs, OptimizeWrite, V-Order, Retention
enable_extended_stats_session = False #: Adds additional write overhead that requires measurement; enable only for join-heavy Gold loads

# ------------------------------------------------------------------------------
# 4. MEDALLION LAYER NAMING KEYWORDS — CUSTOM HEURISTIC
# Microsoft Learn describes the medallion pattern, but names do not prove a table's consumer:
# https://learn.microsoft.com/fabric/onelake/onelake-medallion-lakehouse-architecture
# ------------------------------------------------------------------------------
bronze_staging_keywords = "bronze,raw,stg,staging,landing,temp,tmp,incr_sink,perf_sink"
silver_keywords = "silver,cleansed,curated,int,intermediate"
gold_direct_lake_keywords = "gold,mart,semantic,pbi,rpt,report,dim_,fact_"

# ------------------------------------------------------------------------------
# 5. CUSTOM TABLE HEALTH & ANTI-PATTERN THRESHOLDS
# Closest public guidance:
# https://learn.microsoft.com/fabric/data-engineering/tune-file-size
# https://learn.microsoft.com/fabric/data-engineering/table-compaction
# https://learn.microsoft.com/fabric/data-engineering/delta-lake-partitioning
# https://learn.microsoft.com/fabric/data-engineering/liquid-clustering
# Exact thresholds below are configurable framework heuristics, not Microsoft requirements.
# ------------------------------------------------------------------------------
small_file_size_threshold_mb = 64.0         # Flag SMALL_FILE_PROBLEM when avg file size < 64 MB
large_table_small_file_mb = 128.0           # For large tables (>= 10 GB), expect avg file size >= 128 MB
min_files_for_small_file_flag = 50          #: Minimum small-file count before triggering compaction
single_file_large_table_mb = 5120.0         #: Flag > 5 GB tables stored in <= 2 giant files (0 file skipping)
min_table_size_for_hive_part_mb = 1048576.0 #: Do not use Hive partitioning on tables under 1 TB
min_partition_size_threshold_mb = 128.0     #: Each Hive partition should hold >= 128 MB to 1 GB
max_healthy_Optimize_age_days = 7           # Flag active tables with >= 10 writes and no OPTIMIZE in > 7 days
max_healthy_vacuum_age_days = 7             # Flag modified/optimized tables with no VACUUM in > 7 days
stale_table_inactive_days = 90              # Flag tables unmodified for > 90 days as archival candidates
history_commit_scan_limit = 50              # Number of recent _delta_log commits to inspect per table
inspect_physical_file_metadata = True      #: If True, queries _metadata.file_size for min/median/max file MB
max_discovery_workers = 8                   # Number of parallel threads for workspace-wide discovery

# ------------------------------------------------------------------------------
# 6. TABLE MAINTENANCE & CLEANUP GUARDRAILS
# Public guidance: https://learn.microsoft.com/fabric/data-engineering/delta-lake-vacuum
#                  https://learn.microsoft.com/fabric/fundamentals/table-maintenance-optimization
#                  https://learn.microsoft.com/fabric/data-engineering/automated-table-statistics
# ------------------------------------------------------------------------------
vacuum_retain_hours = 168                   # Safety Floor: 168 hours (7 days). Never run 0h in operational!
allow_unsafe_zero_hour_vacuum = False       # Hard safety lock preventing < 168h VACUUM unless explicitly overridden
staging_retention_days = 7                  # Retention window (days) for Bronze/Staging tables
standard_retention_days = 30                # Retention window (days) for Silver/Gold tables
optimize_where_predicate = ""               # Optional partition filter for OPTIMIZE (e.g. "event_date >= current_date() - INTERVAL 7 DAYS")
use_optimize_full = False                   #: True runs OPTIMIZE FULL; False runs incremental Fast Optimize
run_analyze_table_stats = True              #: Refresh CBO catalog statistics after maintenance
max_parallel_tables = 4                     # Number of parallel maintenance workers using FAIR scheduler pools

# ------------------------------------------------------------------------------
# 7. OPTIONAL BUSINESS DATA LIFECYCLE PURGE RULES (CUSTOM FRAMEWORK INPUT)
# Delta DELETE is public; the lifecycle-rule JSON shape is framework-specific.
# ------------------------------------------------------------------------------
# JSON dictionary mapping "lakehouse.schema.table" -> SQL WHERE predicate to delete expired rows before OPTIMIZE/VACUUM
# Example: '{"example_lakehouse.dbo.staging_events": "event_date < current_date() - INTERVAL 90 DAYS"}'
data_lifecycle_purge_rules_json = "{}"

# ------------------------------------------------------------------------------
# 8. AUDIT LOGGING & ONELAKE SHORTCUT GUARDRAILS
# Shortcut API/source concepts:
# https://learn.microsoft.com/rest/api/fabric/articles/onelakecatalog/overview#list-onelake-shortcuts
# https://learn.microsoft.com/fabric/onelake/onelake-shortcuts
# Audit persistence and single-writer safeguards are framework controls.
# ------------------------------------------------------------------------------
persist_audit_to_delta = True               # Batches all audit results and writes ONCE at the end of the run
audit_table_name = "dbo.lakehouse_optimization_audit_log"
scan_orphan_checkpoints_path = "Files/_checkpoints" #: Folder checked for orphaned streaming checkpoints

discover_onelake_shortcuts = True           # Queries Fabric REST API to identify Tables/ and Files/ shortcuts
inspect_files_shortcuts = True              # Inspects file counts & avg sizes inside Files/ shortcut directories
skip_maintenance_on_shortcuts = True        # Single-Writer Rule: Audit shortcuts, but DO NOT run OPTIMIZE/VACUUM on shortcut targets

# 9. IMPLEMENTATION PLAN & CHANGE-CONTROL GUARDRAILS (CUSTOM FRAMEWORK CONTROLS)
require_explicit_plan_approval = True   # Exact table FQN approval is required before non-audit mutation
approved_plan_table_fqns_json = "[]"     # Example: ["lakehouse.dbo.table"]
persist_implementation_plan_to_delta = True
implementation_plan_table_name = "dbo.lakehouse_optimization_implementation_plan"

# 10. SCREENING AND CLASSIFICATION CONTROLS (CUSTOM FRAMEWORK HEURISTICS)
minimum_table_size_mb_for_testing = 10.0
critical_score_threshold = 70
warning_score_threshold = 35


---
### Step 2: Supported Spark/Delta Configurations, Cluster Topology Inspection & Structured Log4j Logger
**What this cell does:**
1. **Configures Structured JSON Logging via JVM Log4j:**
   * Suppresses noisy third-party loggers (`py4j`, `delta`, root `WARNING`) so driver logs don't suffer from log backpressure.
   * Routes structured JSON events (`run_id`, `event`, `ts`, metrics) through **JVM Log4j** (`spark._jvm.org.apache.log4j.LogManager.getLogger("lakehouse.optimizer")`) so logs land in the **persisted Driver Log tab** and flow automatically through the **Fabric Diagnostic Emitter** (`AzureLogIngestion` DCE/DCR) to Azure Log Analytics.
2. **Inspects Live Cluster Topology:**
   * Reads `sc.defaultParallelism`, `spark.executor.cores`, `spark.executor.memory`, and active JVM executor count to calculate total active Spark vCores and the session's **Capacity Unit (CU) burn rate** ($	ext{1 CU} = 	ext{2 Spark vCores}$).
3. **Applies the  "Baseline Configurations" to the Spark Session:**
   * Enables the **Native Execution Engine (`spark.native.enabled = true`)**, **Adaptive Query Execution (AQE)** + **Skew Join**, **Adaptive Target File Size**, **Fast Optimize**, **File-Level Compaction Targets**, **Deletion Vectors default**, **Parallel Snapshot Loading**, and **Cost-Based Optimizer (CBO)**.


In [ ]:
# ==============================================================================
# STEP 2: GOLDEN SPARK/DELTA CONFIGS, CLUSTER TOPOLOGY & STRUCTURED LOG4J SETUP
# ==============================================================================

import json
import time
import uuid
import logging
import urllib.request
import pandas as pd
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display as ipy_display, HTML
from pyspark.sql import SparkSession, DataFrame, Row, Observation
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.utils import AnalysisException

# UI safety: an individually executed cell may not have received values from the
# parameter cell. Preserve explicit parameter values, but use the safe defaults
# required by this session-configuration cell when they are absent.
run_mode = globals().get("run_mode", "audit_only")
apply_golden_session_configs = globals().get("apply_golden_session_configs", True)
apply_nee_session_config = globals().get("apply_nee_session_config", False)
enable_extended_stats_session = globals().get("enable_extended_stats_session", False)
parameter_cell_was_skipped = any(name not in globals() for name in (
    "scan_scope", "target_lakehouse_name", "small_file_size_threshold_mb",
    "require_explicit_plan_approval"
))
if parameter_cell_was_skipped:
    print("NOTE: Run Step 1: Runtime Parameters before later cells to use full controls; using safe session defaults for this cell.")

# Obtain the active SparkSession and generate a unique correlation ID (run_id) for this execution
spark = SparkSession.builder.getOrCreate()
RUN_ID = str(uuid.uuid4())[:12]
RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).replace(tzinfo=None)

# ------------------------------------------------------------------------------
# 2.1 Configure Structured JVM Log4j + Python Logger (Fabric optimization
# ------------------------------------------------------------------------------
# Rule of thumb from: Set root logger to WARNING, silence noisy libraries,
# and emit 1 structured JSON log line per stage/table rather than per-row print() statements.
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("delta").setLevel(logging.WARNING)

py_logger = logging.getLogger("lakehouse.optimizer")
py_logger.setLevel(logging.INFO)

try:
    # Route to JVM Log4j so messages appear in the Fabric Monitoring Hub -> Driver Logs tab
    # and flow into Azure Log Analytics when Diagnostic Emitter is configured.
    jvm_log4j = spark._jvm.org.apache.log4j.LogManager.getLogger("lakehouse.optimizer")
except Exception:
    jvm_log4j = None

def log_event(event: str, level: str = "INFO", **fields):
    """
    Emits a single structured JSON log event containing run_id, app_id, timestamp,
    and custom key-value metrics to both JVM Log4j and notebook stdout.
    """
    payload = {
        "ts": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "level": level,
        "event": event,
        "run_id": RUN_ID,
        "app_id": spark.conf.get("spark.app.id", "unknown"),
        "run_mode": run_mode,
        **fields
    }
    msg = json.dumps(payload, default=str)
    if jvm_log4j is not None:
        if level == "ERROR":
            jvm_log4j.error(msg)
        elif level == "WARN":
            jvm_log4j.warn(msg)
        else:
            jvm_log4j.info(msg)
    print(f"[{payload['ts']}] [{level}] {event} | {msg}")

# ------------------------------------------------------------------------------
# 2.2 Inspect Live Spark Cluster Topology & CU Footprint (Fabric optimization
# ------------------------------------------------------------------------------
sc = spark.sparkContext
default_parallelism = sc.defaultParallelism
executor_memory = spark.conf.get("spark.executor.memory", "unknown")
executor_cores = int(spark.conf.get("spark.executor.cores", "4") or 4)
driver_memory = spark.conf.get("spark.driver.memory", "unknown")

try:
    # getExecutorMemoryStatus() includes all active executors + 1 for the driver node
    total_jvm_nodes = sc._jsc.sc().getExecutorMemoryStatus().size()
    active_executors = max(1, total_jvm_nodes - 1)
except Exception:
    active_executors = max(1, default_parallelism // max(1, executor_cores))

# Fabric optimization: 1 Fabric Capacity Unit (CU) = 2 Spark vCores
estimated_spark_vcores = active_executors * executor_cores
estimated_cu_footprint = estimated_spark_vcores / 2.0

# ------------------------------------------------------------------------------
# 2.3 Apply Fabric optimization Golden Spark & Delta Session Configurations
# ------------------------------------------------------------------------------
BASELINE_SESSION_CONFIGS = {
    # NEE is opt-in below because the validation run showed substantial overhead on a tiny table.

    # Module 1D & 5B: Enable Adaptive Query Execution (AQE) & Skew Join splitting
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes": "128m",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes": "256m",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor": "5",

    # Module 3G: Delta Lake Baseline Configurations for Runtime 1.3 & 2.0
    "spark.microsoft.delta.targetFileSize.adaptive.enabled": "true", # Auto-scales file sizes (32MB -> 1GB)
    "spark.microsoft.delta.optimize.fast.enabled": "true",           # Skips suboptimal compaction bins (workload-dependent)
    "spark.microsoft.delta.optimize.fileLevelTarget.enabled": "true",# File-level compaction targets
    "spark.databricks.delta.properties.defaults.enableDeletionVectors": "true", # Merge-on-Read bitmaps
    "spark.microsoft.delta.snapshot.driverMode.enabled": "true",     # Faster driver snapshot construction
    "spark.microsoft.delta.parallelSnapshotLoading.enabled": "true", # Parallel snapshot loading

    # Module 3D: Optimize Write pre-write bin size (must be a MB ByteString!)
    "spark.databricks.delta.optimizeWrite.binSize": "128M",
    "spark.databricks.delta.optimizeWrite.partitioned.enabled": "true",

    # Module 3E: Enable Cost-Based Optimizer (CBO) & Join Reordering
    "spark.sql.cbo.enabled": "true",
    "spark.sql.cbo.joinReorder.enabled": "true",
    "spark.sql.cbo.joinReorder.dp.threshold": "12",

    # Module 3G: Keep V-Order OFF by default at session level so Bronze/Silver writes
    # don't pay a workload-dependent write overhead; enable V-Order per-table on Gold Direct Lake tables!
    "spark.sql.parquet.vorder.default": "false"
}

if apply_nee_session_config:
    BASELINE_SESSION_CONFIGS["spark.native.enabled"] = "true"

if enable_extended_stats_session:
    BASELINE_SESSION_CONFIGS["spark.microsoft.delta.stats.collect.extended"] = "true"

applied_configs = {}
skipped_configs = {}

if apply_golden_session_configs:
    for cfg_key, cfg_val in BASELINE_SESSION_CONFIGS.items():
        try:
            spark.conf.set(cfg_key, cfg_val)
            applied_configs[cfg_key] = spark.conf.get(cfg_key)
        except Exception as e:
            skipped_configs[cfg_key] = str(e)

log_event(
    "session_initialized",
    spark_version=spark.version,
    default_parallelism=default_parallelism,
    active_executors=active_executors,
    executor_cores=executor_cores,
    executor_memory=executor_memory,
    driver_memory=driver_memory,
    estimated_spark_vcores=estimated_spark_vcores,
    estimated_cu_burn_rate_per_hr=estimated_cu_footprint,
    golden_configs_applied=len(applied_configs),
    golden_configs_skipped=len(skipped_configs)
)


---
### Step 3A: Workspace-Wide Lakehouse, OneLake Shortcut & Table Catalog Discovery
**What this cell does:**
1. **`discover_workspace_lakehouses()`:** Calls the Fabric Core REST API (`GET /v1/workspaces/{workspaceId}/items?type=Lakehouse`) to find **every Lakehouse in the current workspace** (e.g., all discovered Lakehouses in the current workspace) without requiring you to attach them manually in the UI.
2. **`discover_shortcuts_for_lakehouse()`:** Calls `GET /v1/workspaces/{workspaceId}/items/{lakehouseId}/shortcuts` for each Lakehouse to build a complete map of:
   * **`Tables/` Shortcuts** (e.g., shortcuts pointing to another OneLake Lakehouse, Azure Data Lake Storage Gen2, Amazon S3, Google Cloud Storage, or Dataverse).
   * **`Files/` Shortcuts** (unmanaged folder shortcuts).
3. **`discover_tables_in_lakehouse()`:** Walks each Lakehouse's `abfss://{workspaceId}@onelake.dfs.fabric.microsoft.com/{lakehouseId}/Tables` folder to enumerate every Delta table—automatically supporting both **standard Lakehouses** (`Tables/<table_name>`) and **schema-enabled Lakehouses** (`Tables/<schema_name>/<table_name>`).


In [ ]:
# ==============================================================================
# STEP 3A: WORKSPACE-WIDE LAKEHOUSE, SHORTCUT & TABLE CATALOG DISCOVERY
# ==============================================================================

# Helper function to convert comma-separated parameter strings into clean lowercase Python lists
def parse_csv_list(val: str) -> list:
    return [x.strip().lower() for x in (val or "").split(",") if x.strip()]

# Parse all filter lists from Step 1 Parameters
bronze_kw = parse_csv_list(bronze_staging_keywords)
silver_kw = parse_csv_list(silver_keywords)
gold_kw = parse_csv_list(gold_direct_lake_keywords)
lh_include_set = set(parse_csv_list(target_lakehouse_name))
lh_exclude_set = set(parse_csv_list(lakehouse_exclude_list))
include_set = set(parse_csv_list(table_include_list))
exclude_set = set(parse_csv_list(table_exclude_list))
target_schema_list = parse_csv_list(target_schemas)

# ------------------------------------------------------------------------------
# Function 1: Discover All Lakehouses in the Current Workspace via Fabric REST API
# ------------------------------------------------------------------------------
def discover_workspace_lakehouses() -> tuple:
    """
    Queries GET /v1/workspaces/{wsId}/items?type=Lakehouse using the notebook's
    delegated Entra token to list all Lakehouses in the workspace and construct
    their explicit OneLake ABFS root URIs (abfss://{wsId}@onelake.dfs.fabric.microsoft.com/{lhId}).
    """
    import notebookutils
    rt_ctx = notebookutils.runtime.context
    ws_id = rt_ctx.get("currentWorkspaceId")
    default_lh_id = rt_ctx.get("defaultLakehouseId")
    default_lh_name = rt_ctx.get("defaultLakehouseName")

    default_ctx = {
        "workspace_id": ws_id,
        "lakehouse_id": default_lh_id,
        "lakehouse_name": default_lh_name,
        "abfs_root": f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{default_lh_id}" if (ws_id and default_lh_id) else None
    }

    lakehouses = []
    try:
        # Acquire Fabric API bearer token via notebookutils
        token = notebookutils.credentials.getToken("pbi")
        headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json", "x-ms-fabric-skill": "spark-cli"}
        url = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items?type=Lakehouse"
        all_ws_lhs = []

        # Follow continuationUri pagination if the workspace has many Lakehouses
        while url:
            req = urllib.request.Request(url, headers=headers, method="GET")
            with urllib.request.urlopen(req, timeout=30) as resp:
                data = json.loads(resp.read().decode("utf-8"))
            all_ws_lhs.extend(data.get("value", []))
            url = data.get("continuationUri")

        # Filter Lakehouses based on scan_scope, target_lakehouse_name, and lakehouse_exclude_list
        for item in all_ws_lhs:
            lh_id = item.get("id")
            lh_name = item.get("displayName") or ""
            if not lh_id or not lh_name:
                continue
            if lh_name.lower() in lh_exclude_set or lh_id.lower() in lh_exclude_set:
                continue

            if scan_scope == "current_lakehouse" and not lh_include_set:
                if default_lh_id and lh_id.lower() != default_lh_id.lower():
                    continue
            elif lh_include_set:
                if lh_name.lower() not in lh_include_set and lh_id.lower() not in lh_include_set:
                    continue

            lakehouses.append({
                "workspace_id": ws_id,
                "lakehouse_id": lh_id,
                "lakehouse_name": lh_name,
                "description": item.get("description") or "",
                "abfs_root": f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{lh_id}"
            })

        # If no default Lakehouse was pinned to the notebook, use the first discovered Lakehouse as fallback context
        if not default_ctx["lakehouse_id"] and lakehouses:
            default_ctx = dict(lakehouses[0])

    except Exception as e:
        log_event("workspace_lakehouse_discovery_warning", level="WARN", error=str(e))
        if default_lh_id:
            lakehouses.append(default_ctx)

    return default_ctx, lakehouses

# ------------------------------------------------------------------------------
# Function 2: Discover All OneLake Shortcuts Inside a Lakehouse (Fabric optimization
# ------------------------------------------------------------------------------
def discover_shortcuts_for_lakehouse(lh_meta: dict) -> tuple:
    """
    Calls GET /v1/workspaces/{wsId}/items/{lhId}/shortcuts to identify every
    Table shortcut and Files/ shortcut, extracting its target storage type
    (OneLake, AdlsGen2, AmazonS3, GoogleCloudStorage, Dataverse) and target URI.
    """
    table_shortcuts = {}
    file_shortcuts = []
    if not discover_onelake_shortcuts:
        return table_shortcuts, file_shortcuts

    ws_id = lh_meta["workspace_id"]
    lh_id = lh_meta["lakehouse_id"]
    lh_name = lh_meta["lakehouse_name"]
    try:
        import notebookutils
        token = notebookutils.credentials.getToken("pbi")
        headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json", "x-ms-fabric-skill": "spark-cli"}
        url = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items/{lh_id}/shortcuts"

        while url:
            req = urllib.request.Request(url, headers=headers, method="GET")
            with urllib.request.urlopen(req, timeout=30) as resp:
                data = json.loads(resp.read().decode("utf-8"))

            for sc_item in data.get("value", []):
                sc_path = (sc_item.get("path") or "").strip("/")
                sc_name = sc_item.get("name") or ""
                target = sc_item.get("target") or {}
                t_type = target.get("type") or "Unknown"

                # Format a human-readable target path based on the shortcut target provider
                t_loc = ""
                if target.get("oneLake"):
                    ol = target["oneLake"]
                    t_loc = f"OneLake://{ol.get('workspaceId')}/{ol.get('itemId')}/{ol.get('path')}"
                elif target.get("adlsGen2"):
                    adls = target["adlsGen2"]
                    t_loc = f"{adls.get('location')}{adls.get('subpath')}"
                elif target.get("amazonS3"):
                    s3 = target["amazonS3"]
                    t_loc = f"{s3.get('location')}{s3.get('subpath')}"
                elif target.get("googleCloudStorage"):
                    gcs = target["googleCloudStorage"]
                    t_loc = f"{gcs.get('location')}{gcs.get('subpath')}"
                elif target.get("dataverse"):
                    dv = target["dataverse"]
                    t_loc = f"Dataverse://{dv.get('environmentDomain')}/{dv.get('tableName')}"
                else:
                    t_loc = json.dumps(target)

                sc_info = {
                    "lakehouse_name": lh_name,
                    "lakehouse_id": lh_id,
                    "abfs_root": lh_meta["abfs_root"],
                    "name": sc_name,
                    "path": sc_path,
                    "full_path": f"{sc_path}/{sc_name}".strip("/"),
                    "target_type": t_type,
                    "target_location": t_loc,
                    "target_workspace_id": (target.get("oneLake") or {}).get("workspaceId"),
                    "target_item_id": (target.get("oneLake") or {}).get("itemId"),
                    "target_path": (target.get("oneLake") or {}).get("path")
                                }

                # Separate Tables/ shortcuts from unmanaged Files/ shortcuts
                path_parts = [p for p in sc_path.split("/") if p]
                if path_parts and path_parts[0].lower() == "tables":
                    if len(path_parts) >= 2:
                        schema_key = f"{path_parts[1].lower()}.{sc_name.lower()}"
                        table_shortcuts[schema_key] = sc_info
                    table_shortcuts[sc_name.lower()] = sc_info
                    table_shortcuts[f"dbo.{sc_name.lower()}"] = sc_info
                else:
                    file_shortcuts.append(sc_info)

            url = data.get("continuationUri")
    except Exception as exc:
        log_event("shortcut_discovery_warning", level="WARN", lakehouse=lh_name, error=str(exc))
    return table_shortcuts, file_shortcuts

# ------------------------------------------------------------------------------
# Function 3: Enumerate All Delta Table Paths in a Lakehouse via OneLake ABFS
# ------------------------------------------------------------------------------
def discover_tables_in_lakehouse(lh_meta: dict, sc_map: dict) -> list:
    """
    Lists the `Tables/` directory of a Lakehouse in OneLake ABFS.
    Automatically detects:
      - Standard Lakehouses       : Tables/<table_name>/_delta_log
      - Schema-Enabled Lakehouses : Tables/<schema_name>/<table_name>/_delta_log
    Returns a list of table metadata dicts with `sql_ref = delta.`abfss://...`` so Spark
    can inspect and maintain tables across ANY Lakehouse without switching default Lakehouse!
    """
    import notebookutils
    lh_name = lh_meta["lakehouse_name"]
    lh_id = lh_meta["lakehouse_id"]
    abfs_root = lh_meta["abfs_root"]
    tables_uri = f"{abfs_root}/Tables"
    discovered = []
    seen = set()

    try:
        top_entries = notebookutils.fs.ls(tables_uri)
        for entry in top_entries:
            if not entry.isDir:
                continue
            dir_name = entry.name.strip("/")
            if not dir_name or dir_name.startswith("_"):
                continue

            # Peek 1 level inside the directory to determine if it is a Delta table or a Schema folder
            try:
                child_entries = notebookutils.fs.ls(entry.path)
            except Exception:
                child_entries = []

            has_delta_log = any(c.name.strip("/") == "_delta_log" for c in child_entries)
            has_parquet = any(c.name.endswith(".parquet") for c in child_entries)

            if has_delta_log or has_parquet or (dir_name.lower() in sc_map):
                # Case A: Standard Lakehouse Table -> Tables/<table_name>
                s_name = "dbo"
                t_name = dir_name
                if target_schema_list and s_name not in target_schema_list and lh_name.lower() not in target_schema_list:
                    continue
                fqn = f"{lh_name}.{s_name}.{t_name}"
                short_fqn = f"{s_name}.{t_name}"
                if fqn.lower() in exclude_set or short_fqn.lower() in exclude_set or t_name.lower() in exclude_set:
                    continue
                if include_set and not ({fqn.lower(), short_fqn.lower(), t_name.lower()} & include_set):
                    continue
                if fqn.lower() not in seen:
                    seen.add(fqn.lower())
                    discovered.append({
                        "lakehouse_name": lh_name,
                        "lakehouse_id": lh_id,
                        "schema_name": s_name,
                        "table_name": t_name,
                        "table_fqn": fqn,
                        "sql_ref": f"delta.`{entry.path.rstrip('/')}`",
                        "table_path": entry.path.rstrip("/"),
                        "sc_meta": sc_map.get(short_fqn.lower()) or sc_map.get(t_name.lower())
                    })
            else:
                # Case B: Schema-Enabled Lakehouse -> Tables/<schema_name>/<table_name>
                s_name = dir_name
                if target_schema_list and s_name.lower() not in target_schema_list:
                    continue
                for sub in child_entries:
                    if not sub.isDir:
                        continue
                    t_name = sub.name.strip("/")
                    if not t_name or t_name.startswith("_"):
                        continue
                    fqn = f"{lh_name}.{s_name}.{t_name}"
                    short_fqn = f"{s_name}.{t_name}"
                    if fqn.lower() in exclude_set or short_fqn.lower() in exclude_set or t_name.lower() in exclude_set:
                        continue
                    if include_set and not ({fqn.lower(), short_fqn.lower(), t_name.lower()} & include_set):
                        continue
                    if fqn.lower() not in seen:
                        seen.add(fqn.lower())
                        discovered.append({
                            "lakehouse_name": lh_name,
                            "lakehouse_id": lh_id,
                            "schema_name": s_name,
                            "table_name": t_name,
                            "table_fqn": fqn,
                            "sql_ref": f"delta.`{sub.path.rstrip('/')}`",
                            "table_path": sub.path.rstrip("/"),
                            "sc_meta": sc_map.get(short_fqn.lower()) or sc_map.get(t_name.lower())
                        })
    except Exception as e:
        log_event("lakehouse_table_ls_warning", level="WARN", lakehouse=lh_name, error=str(e))

    return discovered

# ------------------------------------------------------------------------------
# Execute Workspace-Wide Catalog & Shortcut Enumeration in Parallel
# ------------------------------------------------------------------------------
LH_CONTEXT, WORKSPACE_LAKEHOUSES = discover_workspace_lakehouses()
print(f"✅ Discovered {len(WORKSPACE_LAKEHOUSES)} Lakehouse(s) in workspace {LH_CONTEXT.get('workspace_id')} (scan_scope='{scan_scope}')")

all_candidate_tables = []
files_shortcuts_list = []

def discover_single_lakehouse_assets(lh_meta: dict) -> tuple:
    sc_map, f_scs = discover_shortcuts_for_lakehouse(lh_meta)
    tbls = discover_tables_in_lakehouse(lh_meta, sc_map)
    return tbls, f_scs

disc_workers = max(1, min(int(max_discovery_workers), 12))
with ThreadPoolExecutor(max_workers=disc_workers) as pool:
    lh_futures = [pool.submit(discover_single_lakehouse_assets, lh) for lh in WORKSPACE_LAKEHOUSES]
    for fut in as_completed(lh_futures):
        t_list, f_list = fut.result()
        all_candidate_tables.extend(t_list)
        files_shortcuts_list.extend(f_list)

log_event(
    "workspace_catalog_enumerated",
    lakehouses_scanned=len(WORKSPACE_LAKEHOUSES),
    total_candidate_tables=len(all_candidate_tables),
    total_file_shortcuts=len(files_shortcuts_list)
)


---
### Step 3B: Deep Table Profiler & Heuristics Evaluator
**What this cell does:**
For every table discovered in Step 3A, `profile_single_table()` runs 4 diagnostic inspections and evaluates **optimization rules**:
1. **`DESCRIBE DETAIL` Inspection:** Extracts `numFiles`, `sizeInBytes`, `avg_file_mb`, `partitionColumns`, `clusteringColumns` (Liquid Clustering), `minReaderVersion`, `minWriterVersion`, and Delta table properties (`Deletion Vectors`, `V-Order`, `Optimize Write`, `Auto Compaction`, `Column Mapping`, `Log/Deleted File Retention`).
2. **Schema & Column Position Inspection:** Checks whether wide string/JSON columns are sitting inside the **first 32 columns** (`delta.dataSkippingNumIndexedCols = 32`) where they slow down write-time stats collection.
3. **`DESCRIBE HISTORY` Transaction Log Audit:** Scans recent commits to count 7-day writes, `MERGE`/`UPDATE`/`DELETE` churn, tiny `<= 100 row` appends (the **"Delta as a Log Store" anti-pattern**), and days since the last `OPTIMIZE` and `VACUUM`.
4. **11 Heuristics & Priority Scoring:** Assigns flags (`SMALL_FILE_PROBLEM`, `SINGLE_LARGE_FILE_NO_SKIPPING`, `HIVE_PARTITIONED_UNDER_1TB`, `OVER_PARTITIONED_SMALL_FILES`, `UNCLUSTERED_LARGE_TABLE`, `MUTABLE_TABLE_WITHOUT_DV`, `NEEDS_MAINTENANCE_OPTIMIZE`, `UNVACUUMED_STORAGE_BLOAT`, `LOG_STORE_SMALL_COMMIT_ABUSE`, `GOLD_MISSING_VORDER`, `BRONZE_UNNECESSARY_VORDER_TAX`, `STATS_COLUMN_ORDER_SUBOPTIMAL`, `STALE_OR_TEMP_TABLE_CANDIDATE`) and computes a `priority_score` (`CRITICAL >= 50`, `WARNING >= 20`, `HEALTHY < 20`).


In [ ]:
# ==============================================================================
# STEP 3B: DEEP TABLE PROFILER & Fabric optimization HEURISTICS EVALUATOR
# ==============================================================================

# Helper: Classify table into Bronze/Staging, Silver/Curated, or Gold/Direct Lake based on naming conventions
def infer_table_layer(lh_name: str, schema_name: str, table_name: str) -> str:
    full = f"{lh_name}.{schema_name}.{table_name}".lower()
    if any(k in full for k in gold_kw):
        return "GOLD_DIRECT_LAKE"
    if any(k in full for k in bronze_kw):
        return "BRONZE_STAGING"
    if any(k in full for k in silver_kw):
        return "SILVER_CURATED"
    return "GENERAL"

def profile_single_table(tbl_info: dict) -> dict:
    """
    Profiles a single Delta table (native or shortcut) and evaluates all 11 Fabric optimization
    optimization heuristics to produce a health status, priority score, and action plan.
    """
    t0 = time.time()
    lh_name = tbl_info["lakehouse_name"]
    lh_id = tbl_info["lakehouse_id"]
    schema_name = tbl_info["schema_name"]
    table_name = tbl_info["table_name"]
    fqn = tbl_info["table_fqn"]
    quoted_fqn = tbl_info["sql_ref"]  # Uses `delta.`abfss://...`` so it works across any Lakehouse
    layer = infer_table_layer(lh_name, schema_name, table_name)
    flags = []
    recommendations = []

    # Check if this table was identified as a OneLake Shortcut in Step 3A
    sc_meta = tbl_info.get("sc_meta")
    is_shortcut = sc_meta is not None
    shortcut_type = sc_meta["target_type"] if sc_meta else "NONE"
    shortcut_target = sc_meta["target_location"] if sc_meta else ""
    metric_status = "NATIVE_DESCRIBE_DETAIL"
    metric_verification_note = "Native table metrics read from DESCRIBE DETAIL."
    consumer_num_files = -1
    consumer_size_mb = 0.0
    owner_num_files = -1
    owner_size_mb = 0.0
    metrics_verified = not is_shortcut

    try:
        # Tag the Spark job description so it is easy to spot in the Spark UI (Fabric optimization
        spark.sparkContext.setJobDescription(f"Discovery Audit: {fqn}")

        # ----------------------------------------------------------------------
        # 1. Physical Metadata Inspection via DESCRIBE DETAIL
        # ----------------------------------------------------------------------
        detail_row = spark.sql(f"DESCRIBE DETAIL {quoted_fqn}").first()
        detail = detail_row.asDict() if detail_row else {}
        fmt = (detail.get("format") or "").lower()
        table_location = str(detail.get("location") or "")

        # Fallback check: if physical location points to an external storage account, mark as shortcut
        if not is_shortcut and any(ext in table_location.lower() for ext in (".dfs.core.windows.net", "s3://", "s3a://", "gs://")):
            is_shortcut = True
            shortcut_type = "ExternalPath"
            shortcut_target = table_location

        if fmt != "delta":
            return {
                "lakehouse_name": lh_name, "lakehouse_id": lh_id,
                "table_fqn": fqn, "sql_ref": quoted_fqn, "schema_name": schema_name, "table_name": table_name,
                "layer": layer, "format": fmt, "is_shortcut": is_shortcut,
                "shortcut_type": shortcut_type, "shortcut_target": shortcut_target,
                "health_status": "SKIPPED_NON_DELTA", "evidence_tier": "INFORMATIONAL", "test_plan": "Not a Delta table.", "adaptive_target_file_size": "n/a", "priority_score": 0,
                "size_mb": 0.0, "num_files": 0, "avg_file_mb": 0.0,
                "min_file_mb": 0.0, "median_file_mb": 0.0, "max_file_mb": 0.0,
                "metric_status": metric_status, "metric_verification_note": metric_verification_note,
                "consumer_num_files": consumer_num_files, "consumer_size_mb": consumer_size_mb,
                "owner_num_files": owner_num_files, "owner_size_mb": owner_size_mb,
                "partition_cols": "", "clustering_cols": "", "candidate_key_cols": "", "total_cols": 0,
                "dv_enabled": False, "vorder_enabled": "n/a", "optimize_write": "n/a",
                "auto_compact": "n/a", "column_mapping": "n/a", "log_retention": "",
                "deleted_file_retention": "", "min_reader_version": 0, "min_writer_version": 0,
                "write_commits_7d": 0, "merge_delete_commits_7d": 0, "small_append_commits": 0,
                "days_since_optimize": -1, "days_since_vacuum": -1, "days_since_modified": -1,
                "needs_optimize": False, "needs_vacuum": False, "needs_analyze": False,
                "flags": "NON_DELTA", "recommendations": "Convert to Delta format.",
            "remedy_sql": f"-- Convert {fqn} to Delta format",
                "scan_duration_s": round(time.time() - t0, 2)
            }

        # Calculate file counts, total MB, and average file size MB
        num_files = int(detail.get("numFiles") or 0)
        size_bytes = int(detail.get("sizeInBytes") or 0)
        consumer_num_files = num_files
        consumer_size_mb = round(size_bytes / (1024.0 * 1024.0), 2)
        metric_status = "SHORTCUT_CONSUMER_ESTIMATE_UNVERIFIED" if is_shortcut else "NATIVE_DESCRIBE_DETAIL"
        metric_verification_note = "Consumer DESCRIBE DETAIL only; shortcut metrics require reconciliation." if is_shortcut else "Native table DESCRIBE DETAIL."
        metrics_verified = not is_shortcut

        owner_ref = ""
        if is_shortcut and sc_meta:
            endpoint_host = str(notebookutils.conf.get("trident.onelake.endpoint")).replace("https://", "").replace("http://", "").rstrip("/")
            if sc_meta.get("target_workspace_id") and sc_meta.get("target_item_id") and sc_meta.get("target_path"):
                owner_ref = (
                    f"delta.`abfss://{sc_meta['target_workspace_id']}@{endpoint_host}/"
                    f"{sc_meta['target_item_id']}/{sc_meta['target_path'].strip('/')}`"
                )
            elif shortcut_type.lower() == "adlsgen2" and shortcut_target.startswith("https://"):
                parts = shortcut_target.replace("https://", "", 1).split("/")
                if len(parts) >= 3:
                    account_host, container, rest = parts[0], parts[1], "/".join(parts[2:])
                    owner_ref = f"delta.`abfss://{container}@{account_host}/{rest}`"

        if is_shortcut and owner_ref:
            try:
                owner_detail = spark.sql(f"DESCRIBE DETAIL {owner_ref}").first().asDict()
                owner_num_files = int(owner_detail.get("numFiles") or 0)
                owner_size_bytes = int(owner_detail.get("sizeInBytes") or 0)
                owner_size_mb = round(owner_size_bytes / (1024.0 * 1024.0), 2)
                if owner_num_files >= 0 and owner_size_bytes >= 0:
                    num_files = owner_num_files
                    size_bytes = owner_size_bytes
                    metrics_verified = True
                    metric_status = "OWNER_TARGET_VERIFIED"
                    metric_verification_note = f"Owner target {shortcut_target} matched through direct DESCRIBE DETAIL; consumer reported {consumer_num_files} files / {consumer_size_mb:.2f} MB."
            except Exception as exc:
                metric_verification_note = f"Owner-target reconciliation failed; using consumer metadata only: {type(exc).__name__}: {exc}"

        size_mb = round(size_bytes / (1024.0 * 1024.0), 2)
        avg_file_mb = round(size_mb / num_files, 2) if num_files > 0 else 0.0
        part_cols = list(detail.get("partitionColumns") or [])
        cluster_cols = list(detail.get("clusteringColumns") or [])
        props = dict(detail.get("properties") or {})
        min_reader = int(detail.get("minReaderVersion") or 1)
        min_writer = int(detail.get("minWriterVersion") or 2)
        last_modified = detail.get("lastModified")
        days_since_modified = (RUN_TIMESTAMP_UTC - last_modified).days if last_modified else -1

        # Parse Delta table properties from DESCRIBE DETAIL
        dv_enabled = str(props.get("delta.enableDeletionVectors", "false")).lower() == "true"
        vorder_prop = str(props.get("delta.parquet.vorder.enabled", "unset")).lower()
        opt_write_prop = str(props.get("delta.autoOptimize.optimizeWrite", "unset")).lower()
        auto_compact_prop = str(props.get("delta.autoOptimize.autoCompact", "unset")).lower()
        log_ret_prop = props.get("delta.logRetentionDuration", "default (30 days)")
        del_ret_prop = props.get("delta.deletedFileRetentionDuration", "default (7 days)")
        col_mapping = props.get("delta.columnMapping.mode", "none")
        indexed_cols_num = int(props.get("delta.dataSkippingNumIndexedCols", "32") or 32)
        stats_cols_prop = props.get("delta.dataSkippingStatsColumns", "")
        adaptive_target_file_size = props.get("delta.targetFileSize.adaptive", "unset")

        # ----------------------------------------------------------------------
        # 2. Schema & File-Skipping Column Order Inspection
        # ----------------------------------------------------------------------
        # Delta collects min/max stats on the first 32 columns by default.
        # Wide string/JSON columns in the first 32 positions waste write CPU.
        tbl_schema = spark.sql(f"SELECT * FROM {quoted_fqn} LIMIT 0").schema
        total_cols = len(tbl_schema.fields)
        first_n_fields = tbl_schema.fields[:indexed_cols_num]
        string_cols_in_first_32 = [
            f.name for f in first_n_fields
            if isinstance(f.dataType, T.StringType) and any(
                kw in f.name.lower() for kw in ("json", "payload", "body", "xml", "blob", "comment", "description", "notes", "raw", "log", "text")
            )
        ]
        # Candidate columns are scaffolding for tests only. They must be confirmed
        # against real workload filters/joins before any CLUSTER BY or CBO action.
        candidate_key_cols = [
            f.name for f in tbl_schema.fields
            if isinstance(f.dataType, (T.DateType, T.TimestampType, T.IntegerType, T.LongType))
            or any(k in f.name.lower() for k in ("date", "dt", "_id", "_sk", "_key", "region", "status", "type", "code"))
        ][:3]
        candidate_cols_outside_stats = [
            c for c in candidate_key_cols
            if c in [f.name for f in tbl_schema.fields[indexed_cols_num:]]
            and not stats_cols_prop
        ]

        # ----------------------------------------------------------------------
        # 3. Transaction History Audit via DESCRIBE HISTORY
        # ----------------------------------------------------------------------
        hist_rows = spark.sql(f"DESCRIBE HISTORY {quoted_fqn} LIMIT {int(history_commit_scan_limit)}").collect()
        write_commits_7d = 0
        small_append_commits = 0
        merge_delete_update_commits_7d = 0
        days_since_optimize = 999
        days_since_vacuum = 999
        unoptimized_writes_since_last_opt = 0
        seen_optimize = False

        for h in hist_rows:
            hd = h.asDict()
            op = (hd.get("operation") or "").upper()
            ts = hd.get("timestamp")
            age_days = (RUN_TIMESTAMP_UTC - ts).days if ts else 999
            metrics = hd.get("operationMetrics") or {}

            # Track most recent OPTIMIZE and VACUUM operations
            if op == "OPTIMIZE" and days_since_optimize == 999:
                days_since_optimize = max(0, age_days)
                seen_optimize = True
            elif op.startswith("VACUUM") and days_since_vacuum == 999:
                days_since_vacuum = max(0, age_days)

            # Track write frequency, mutation churn (MERGE/UPDATE/DELETE), and tiny <=100 row commits
            if op in ("WRITE", "STREAMING UPDATE", "APPEND", "MERGE", "UPDATE", "DELETE", "CREATE OR REPLACE TABLE AS SELECT", "REPLACE TABLE AS SELECT"):
                if not seen_optimize:
                    unoptimized_writes_since_last_opt += 1
                if age_days <= 7:
                    write_commits_7d += 1
                if op in ("MERGE", "UPDATE", "DELETE") and age_days <= 7:
                    merge_delete_update_commits_7d += 1

                num_rows_out = metrics.get("numOutputRows") or metrics.get("numTargetRowsInserted")
                if num_rows_out is not None:
                    try:
                        if 0 < int(num_rows_out) <= 100:
                            small_append_commits += 1
                    except Exception:
                        pass

        # ----------------------------------------------------------------------
        # 4. Optional Exact File Size Percentiles via _metadata.file_size
        # ----------------------------------------------------------------------
        min_file_mb = avg_file_mb
        median_file_mb = avg_file_mb
        max_file_mb = avg_file_mb
        if inspect_physical_file_metadata and num_files > 0:
            try:
                meta_row = spark.sql(f"""
                    SELECT
                        COUNT(*) AS physical_files,
                        COALESCE(SUM(file_size), 0) AS physical_bytes,
                        ROUND(MIN(file_size) / 1048576.0, 2) AS min_mb,
                        ROUND(PERCENTILE_APPROX(file_size, 0.5) / 1048576.0, 2) AS med_mb,
                        ROUND(MAX(file_size) / 1048576.0, 2) AS max_mb
                    FROM (
                        SELECT DISTINCT _metadata.file_path AS file_path, _metadata.file_size AS file_size
                        FROM {quoted_fqn}
                    )
                """).first()
            except Exception as exc:
                meta_row = None
                if is_shortcut:
                    metric_verification_note += f" Physical _metadata reconciliation failed: {type(exc).__name__}: {exc}"
            if meta_row:
                physical_files = int(meta_row["physical_files"] or 0)
                physical_bytes = int(meta_row["physical_bytes"] or 0)
                min_file_mb = float(meta_row["min_mb"] or 0.0)
                median_file_mb = float(meta_row["med_mb"] or 0.0)
                max_file_mb = float(meta_row["max_mb"] or 0.0)
                if is_shortcut and not metrics_verified and physical_files > 0:
                    num_files = physical_files
                    size_bytes = physical_bytes
                    size_mb = round(size_bytes / (1024.0 * 1024.0), 2)
                    avg_file_mb = round(size_mb / num_files, 2)
                    metrics_verified = True
                    metric_status = "CONSUMER_PHYSICAL_METADATA_VERIFIED"
                    metric_verification_note = (
                        f"Shortcut consumer snapshot reconciled from _metadata: {physical_files} files / "
                        f"{size_mb:.2f} MB. Direct owner metrics were unavailable; verify owner freshness before action."
                    )

        # ======================================================================
        # 5. EVALUATE THE 11 Fabric optimization OPTIMIZATION HEURISTICS & SCORE THE TABLE
        # ======================================================================
        priority_score = 0
        needs_optimize = False
        needs_vacuum = False
        needs_analyze = False

        # Shortcut Tagging & Cross-Cloud Egress Check
        if is_shortcut:
            flags.append(f"SHORTCUT({shortcut_type})")
            if not metrics_verified:
                flags.append("SHORTCUT_METRICS_UNVERIFIED")
                priority_score += 5
                recommendations.append(
                    f"Shortcut physical metrics are unverified ({metric_status}). "
                    "Do not use file count/size recommendations until the owner target is measured."
                )
            if shortcut_type in ("AmazonS3", "GoogleCloudStorage", "S3Compatible"):
                flags.append("CROSS_CLOUD_SHORTCUT_EGRESS_RISK")
                recommendations.append(
                    f"External {shortcut_type} shortcut ({shortcut_target}): mind cross-cloud egress costs and remote read latency."
                )
            else:
                recommendations.append(
                    f"Shortcut ({shortcut_type} -> {shortcut_target}): Single-Writer Rule applies—run OPTIMIZE/VACUUM in the owning source workspace/platform."
                )

        # Heuristic 1: Small File Problem
        target_min_mb = large_table_small_file_mb if size_mb >= 10240.0 else small_file_size_threshold_mb
        if metrics_verified and num_files >= min_files_for_small_file_flag and avg_file_mb < target_min_mb:
            flags.append("SMALL_FILE_PROBLEM")
            priority_score += 20 if num_files >= 500 else 10
            recommendations.append(
                f"{num_files} files average {avg_file_mb} MB. Treat as a compaction candidate, not proof of query harm. "
                "Confirm a read/commit regression and compare Optimize Write, Auto Compaction, and no-change controls before OPTIMIZE."
            )

        # Heuristic 2: Single / Too Few Giant Files Hurting Selective Skipping
        if metrics_verified and size_mb >= single_file_large_table_mb and 0 < num_files <= 2:
            flags.append("SINGLE_LARGE_FILE_NO_SKIPPING")
            needs_optimize = True
            priority_score += 20
            recommendations.append(
                f"Table is {size_mb} MB in only {num_files} file(s); selective lookups cannot skip files. Enable Adaptive Target File Size & Liquid Clustering."
            )

        # Heuristic 3: Possible over-partitioning.
        # These are migration investigation candidates, not proof that partitioning is harmful.
        if len(part_cols) > 0:
            if size_mb < min_table_size_for_hive_part_mb:
                flags.append("HIVE_PARTITIONED_UNDER_1TB")
                priority_score += 20
                recommendations.append(
                    f"Hive partitioned on {part_cols} but size is {size_mb} MB (< 1 TB,. Consider migrating to Liquid Clustering (CLUSTER BY)."
                )
            if metrics_verified and (len(part_cols) >= 3 or (num_files > 100 and avg_file_mb < min_partition_size_threshold_mb)):
                flags.append("OVER_PARTITIONED_SMALL_FILES")
                needs_optimize = True
                priority_score += 30
                recommendations.append(
                    f"Over-partitioned on {part_cols} (avg file {avg_file_mb} MB < 128 MB,. Migrate via CTAS to Liquid Clustering."
                )

        # Heuristic 4: Unpartitioned & Unclustered Large Table >= 1 GB
        if metrics_verified and len(part_cols) == 0 and len(cluster_cols) == 0 and size_mb >= 1024.0:
            flags.append("UNCLUSTERED_LARGE_TABLE")
            priority_score += 15
            recommendations.append(
                "Table >= 1 GB has neither Liquid Clustering nor Partitioning. Run ALTER TABLE ... CLUSTER BY (1-4 filter/join cols)."
            )

        # Heuristic 5: Unoptimized Writes / High Mutation Churn without Deletion Vectors
        if merge_delete_update_commits_7d > 0 and not dv_enabled:
            flags.append("MUTABLE_TABLE_WITHOUT_DV")
            priority_score += 20
            recommendations.append(
                f"{merge_delete_update_commits_7d} MERGE/UPDATE/DELETE commits in 7d without Deletion Vectors. Enable 'delta.enableDeletionVectors'='true'."
            )
        if unoptimized_writes_since_last_opt >= 10 and (days_since_optimize > max_healthy_Optimize_age_days or merge_delete_update_commits_7d >= 3):
            flags.append("NEEDS_MAINTENANCE_OPTIMIZE")
            needs_optimize = True
            priority_score += 15
            recommendations.append(
                f"{unoptimized_writes_since_last_opt} writes since last OPTIMIZE ({days_since_optimize}d ago). Run OPTIMIZE to compact & purge DVs > 5%."
            )

        # Heuristic 6: Unvacuumed Storage Bloat
        if (days_since_vacuum > max_healthy_vacuum_age_days and (write_commits_7d > 0 or days_since_optimize <= 14)) or (days_since_vacuum == 999 and len(hist_rows) >= 5):
            flags.append("VACUUM_HISTORY_NOT_OBSERVED")
            # Missing VACUUM history is not proof of orphan-file bloat. Keep this
            # as a conditional storage test; maintenance is triggered after an
            # OPTIMIZE/mutation or explicit ownership review.
            needs_vacuum = False
            priority_score += 5
            recommendations.append(
                f"VACUUM was not observed in the inspected history ({days_since_vacuum if days_since_vacuum != 999 else 'outside scanned window'}d). "
                f"Measure physical orphan storage first; do not run VACUUM solely from this signal."
            )

        # Heuristic 7: Delta Misused as Operational Log Store Anti-Pattern
        if small_append_commits >= 15:
            flags.append("LOG_STORE_SMALL_COMMIT_ABUSE")
            needs_optimize = True
            needs_vacuum = False
            priority_score += 35
            recommendations.append(
                f"{small_append_commits} recent tiny commits (<=100 rows). Delta is not a per-event log store! Buffer writes or route logs to Log Analytics / Eventhouse."
            )

        # Heuristic 8: V-Order consumer review.
        # Layer names alone cannot prove Direct Lake consumption. Keep this informational
        # and require an explicit Direct Lake/read-heavy consumer benchmark.
        if layer in ("GOLD_DIRECT_LAKE", "BRONZE_STAGING"):
            flags.append("VORDER_CONSUMER_REVIEW")
            priority_score += 3
            recommendations.append(
                f"V-Order property is '{vorder_prop}'. Confirm actual consumers and benchmark read gain versus write overhead before changing it."
            )

        # Heuristic 9: only flag a suspected filter/join column that is actually outside stats coverage.
        if candidate_cols_outside_stats:
            flags.append("FILTER_COLUMNS_OUTSIDE_STATS")
            priority_score += 8
            recommendations.append(
                f"Candidate workload columns outside indexed stats coverage: {candidate_cols_outside_stats}. "
                "Confirm they are real filters/joins, then test explicit dataSkippingStatsColumns."
            )
        elif string_cols_in_first_32 and not stats_cols_prop:
            flags.append("WIDE_STRING_STATS_REVIEW")
            priority_score += 3
            recommendations.append(
                f"Wide string-like columns {string_cols_in_first_32[:3]} are in the first {indexed_cols_num} columns. "
                "Review write overhead only if these columns are never useful for filtering."
            )

        # Heuristic 10: Stale / Temp / Backup Table Candidate for Archival
        is_temp_named = any(tag in table_name.lower() for tag in ("_tmp", "_temp", "_bak", "_backup", "_old", "test_", "tmp_"))
        if is_temp_named or (days_since_modified >= stale_table_inactive_days and size_mb > 0):
            flags.append("STALE_OR_TEMP_TABLE_CANDIDATE")
            priority_score += 10
            recommendations.append(
                f"Table '{table_name}' (last modified {days_since_modified}d ago, {size_mb} MB) appears stale/temporary. Verify with owner and drop/archive to reclaim OneLake storage."
            )

        # Heuristic 11: Refresh CBO Statistics on Active Silver/Gold Tables
        if run_analyze_table_stats and layer in ("SILVER_CURATED", "GOLD_DIRECT_LAKE") and (needs_optimize or write_commits_7d > 0) and not quoted_fqn.startswith("delta.`"):
            needs_analyze = True

        # Evidence tiers separate physical optimization candidates from metadata-only review.
        strong_physical_evidence = any(
            f in flags for f in (
                "OVER_PARTITIONED_SMALL_FILES",
                "NEEDS_MAINTENANCE_OPTIMIZE",
                "LOG_STORE_SMALL_COMMIT_ABUSE",
            )
        )
        conditional_evidence = any(
            f in flags for f in (
                "SMALL_FILE_PROBLEM",
                "SINGLE_LARGE_FILE_NO_SKIPPING",
                "HIVE_PARTITIONED_UNDER_1TB",
                "UNCLUSTERED_LARGE_TABLE",
                "MUTABLE_TABLE_WITHOUT_DV",
                "VACUUM_HISTORY_NOT_OBSERVED",
                "FILTER_COLUMNS_OUTSIDE_STATS",
                "SHORTCUT_METRICS_UNVERIFIED",
            )
        )
        if size_mb < minimum_table_size_mb_for_testing and not any(
            f in flags for f in ("LOG_STORE_SMALL_COMMIT_ABUSE", "STALE_OR_TEMP_TABLE_CANDIDATE")
        ):
            evidence_tier = "INFORMATIONAL"
        elif strong_physical_evidence:
            evidence_tier = "STRONG_CANDIDATE"
        elif conditional_evidence:
            evidence_tier = "CONDITIONAL_CANDIDATE"
        elif flags:
            evidence_tier = "INFORMATIONAL"
        else:
            evidence_tier = "HEALTHY_CONTROL"

        health_status = (
            "CRITICAL"
            if priority_score >= critical_score_threshold
            else ("WARNING" if priority_score >= warning_score_threshold else "HEALTHY")
        )

        # ----------------------------------------------------------------------
        # 6. Build a Test Plan and Candidate Remedy SQL
        # ----------------------------------------------------------------------
        remedy_commands = []
        suggested_keys_csv = ", ".join(candidate_key_cols) if candidate_key_cols else "col1, col2"

        if is_shortcut and skip_maintenance_on_shortcuts:
            remedy_commands.append(f"-- [SHORTCUT GUARDRAIL] Apply the equivalent remedy to the target owner: {shortcut_target}. Do not run maintenance through the shortcut consumer.")

        # (A) Table Property Remedies (Deletion Vectors, Optimize Write, V-Order, Retention, Stats Columns)
        tbl_props_to_set = {}
        if "MUTABLE_TABLE_WITHOUT_DV" in flags and not dv_enabled:
            tbl_props_to_set["delta.enableDeletionVectors"] = "true"
        if ("SMALL_FILE_PROBLEM" in flags or "LOG_STORE_SMALL_COMMIT_ABUSE" in flags) and opt_write_prop != "true":
            # Candidate only: the validation matrix favored Optimize Write over enabling both controls.
            tbl_props_to_set["delta.autoOptimize.optimizeWrite"] = "true"
        # Auto Compaction is intentionally not auto-prescribed; its post-write cost can outweigh file-count gains.
        # V-Order is never prescribed from naming convention; validate the consumer first.
        if layer == "BRONZE_STAGING" and ("VACUUM_HISTORY_NOT_OBSERVED" in flags or "LOG_STORE_SMALL_COMMIT_ABUSE" in flags):
            tbl_props_to_set["delta.logRetentionDuration"] = f"interval {int(staging_retention_days)} days"
            tbl_props_to_set["delta.deletedFileRetentionDuration"] = f"interval {int(staging_retention_days)} days"
        if "FILTER_COLUMNS_OUTSIDE_STATS" in flags and candidate_key_cols:
            tbl_props_to_set["delta.dataSkippingStatsColumns"] = ",".join(candidate_key_cols)

        if tbl_props_to_set and not (is_shortcut and skip_maintenance_on_shortcuts):
            props_sql_str = ", ".join(f"'{k}' = '{v}'" for k, v in tbl_props_to_set.items())
            remedy_commands.append(f"ALTER TABLE {quoted_fqn} SET TBLPROPERTIES ({props_sql_str});")

        # (B) Physical Data Organization Remedies (Liquid Clustering / Hive Migration)
        if "UNCLUSTERED_LARGE_TABLE" in flags and not (is_shortcut and skip_maintenance_on_shortcuts):
            remedy_commands.append(                f"-- Candidate only: confirm these are real workload filter/join columns. Column order does not affect clustering.\n"                f"ALTER TABLE {quoted_fqn} CLUSTER BY ({suggested_keys_csv});")
        elif ("HIVE_PARTITIONED_UNDER_1TB" in flags or "OVER_PARTITIONED_SMALL_FILES" in flags) and not (is_shortcut and skip_maintenance_on_shortcuts):
            remedy_commands.append(
                f"-- Migrate from Hive Partitioning ({','.join(part_cols)}) to Liquid Clustering: "
                f"CREATE OR REPLACE TABLE {fqn} CLUSTER BY ({suggested_keys_csv}) AS SELECT * FROM {quoted_fqn};"
            )

        # (C) File Compaction & Deletion Vector Purge Remedy (OPTIMIZE)
        if needs_optimize and not (is_shortcut and skip_maintenance_on_shortcuts):
            remedy_commands.append(
                f"SET spark.microsoft.delta.targetFileSize.adaptive.enabled = true; "
                f"SET spark.microsoft.delta.optimize.fast.enabled = true; "
                f"OPTIMIZE {quoted_fqn};"
            )

        # (D) Orphan File Storage Reclamation Remedy (VACUUM)
        if needs_vacuum and not (is_shortcut and skip_maintenance_on_shortcuts):
            remedy_commands.append(f"VACUUM {quoted_fqn} RETAIN {int(vacuum_retain_hours)} HOURS;")

        # (E) Cost-Based Optimizer Statistics Refresh Remedy (ANALYZE TABLE)
        if (needs_analyze or (size_mb >= 500.0 and candidate_key_cols)) and not (is_shortcut and skip_maintenance_on_shortcuts):
            remedy_commands.append(f"ANALYZE TABLE {fqn} COMPUTE STATISTICS FOR COLUMNS {suggested_keys_csv};")

        # (F) Stale / Temporary Table Archival Remedy
        if "STALE_OR_TEMP_TABLE_CANDIDATE" in flags:
            remedy_commands.append(f"-- After owner verification: DROP TABLE IF EXISTS {fqn};")

        test_plan = []
        if "SMALL_FILE_PROBLEM" in flags or "NEEDS_MAINTENANCE_OPTIMIZE" in flags:
            test_plan.append(
                "SMALL-FILE / OPTIMIZE TEST | Objective: prove whether layout change improves real reads and writes. "
                "Variants: (A) current table unchanged, (B) identical clone written with Optimize Write enabled, "
                "(C) controlled incremental OPTIMIZE copy. Query design: use one representative selective query with the table's real WHERE predicates and one full-table aggregate; each query must execute a Spark action and verify identical row count/checksum. "
                "Protocol: execute one warm-up excluded from analysis, then five cold/warm measured repetitions per query and variant; keep row data, schema, pool, parallelism, and session configuration equal. "
                "Capture: min/median/max duration, files read, bytes read, Spark input partitions, table file count/MB, write or OPTIMIZE duration, and Capacity Units when available. "
                "Accept only if correctness matches, median selective/full-read benefit clears the agreed SLA threshold, and write/maintenance/CU regression remains within the approved limit; otherwise keep the current layout."
            )
        if "MUTABLE_TABLE_WITHOUT_DV" in flags:
            test_plan.append(
                "DELETION-VECTOR TEST | Objective: quantify Merge-on-Read trade-offs. Variants: identical DV-off and DV-on clones from the same snapshot. "
                "Operation: replay the same representative UPDATE, DELETE, or MERGE with the same source and predicate. Run it once as setup plus five measured replay cycles on fresh/reset clones. "
                "Capture mutation duration, files/bytes rewritten, generated deletion-vector density, post-mutation read duration, files/bytes read, CU, and row/checksum correctness. "
                "Accept only if mutation cost improves beyond the agreed threshold without violating read latency, cleanup frequency, or correctness requirements; otherwise do not enable Deletion Vectors."
            )
        if "UNCLUSTERED_LARGE_TABLE" in flags or "HIVE_PARTITIONED_UNDER_1TB" in flags or "OVER_PARTITIONED_SMALL_FILES" in flags:
            test_plan.append(
                "CLUSTERING TEST | Objective: prove that candidate layout keys match real workload predicates. "
                "Inputs: identify the top real filter/join columns from query history; do not use column-name guesses as final keys. "
                "Variants: (A) equal unclustered/current-layout control, (B) equal Liquid-Clustered clone with candidate keys, and (C) existing Hive-partitioned layout if applicable. "
                "Apply OPTIMIZE only to the clustered variant to materialize its layout. Queries: use at least one selective equality/range query, one multi-predicate query, and one full-table aggregate that users actually run. "
                "Protocol: one excluded warm-up plus five measured executions for each query/variant; optionally repeat in a fresh Spark session for cold behavior. "
                "Capture: files skipped, files/bytes read, duration min/median/max, CU, write/OPTIMIZE duration, resulting file count/MB, and clustering-quality metrics where supported. "
                "Acceptance: correctness and schema match; selective workload improves by the approved threshold; full-scan and maintenance regression stay within limits. If keys cannot be tied to recurring predicates, mark INCONCLUSIVE rather than promoting."
            )
        if "VACUUM_HISTORY_NOT_OBSERVED" in flags:
            test_plan.append(
                "VACUUM SAFETY/RECLAMATION TEST | Objective: separate storage cleanup evidence from history age. "
                "First list unreferenced/orphan Parquet candidates and their modification age; verify active readers, time-travel requirements, backup/DR policy, and minimum retention. "
                "Use a clone for destructive proof. Verify current row/checksum baseline, run VACUUM RETAIN 168 HOURS, then verify row/checksum, file count, and active-reader behavior. Record orphan bytes before, bytes removed, command duration, and errors. "
                "Accept only if measurable eligible orphan storage is reclaimed and current data remains readable. Missing VACUUM history by itself never authorizes execution."
            )
        if "FILTER_COLUMNS_OUTSIDE_STATS" in flags:
            test_plan.append(
                "DATA-SKIPPING STATISTICS TEST | Objective: verify that explicit statistics columns improve real predicate skipping. "
                "Variants: identical current-property table and clone with only confirmed workload columns in delta.dataSkippingStatsColumns. "
                "Queries: one equality filter and one range/join predicate using those columns, plus a control query on a non-target column. "
                "Run one excluded warm-up and five measured executions per query/variant. Capture files/bytes read, file-skipping count, duration, CU, write/commit cost, Delta-log overhead, and correctness. "
                "Accept only for demonstrated selective-read benefit without unacceptable write/CU overhead."
            )
        if "VORDER_CONSUMER_REVIEW" in flags:
            test_plan.append(
                "V-ORDER CONSUMER TEST | Objective: test the actual consumer, not table naming. "
                "Variants: identical off/on V-Order tables written with equal data and compute. Consumers: run the actual Spark workload and, when applicable, the real Direct Lake semantic-model/DAX workload. "
                "Protocol: one excluded warm-up plus five measured warm runs; run separate cold-session runs with cleared/new session and report them separately. "
                "Capture table size, write duration, refresh duration, query duration, rows/files/bytes read, CU, cache state, and correctness. "
                "Accept only if the confirmed consumer benefit exceeds the agreed threshold and write/refresh overhead stays within limits; otherwise leave V-Order unchanged."
            )
        if "LOG_STORE_SMALL_COMMIT_ABUSE" in flags:
            test_plan.append(
                "WRITE-PATTERN TEST | Objective: replace per-row/per-event Delta commits with a safe delivery pattern. "
                "Variants: current per-event path versus buffered batch or streaming micro-batch with equal event data and durability semantics. "
                "Run a fixed event volume at least five times. Capture commits, files, end-to-end latency, events/second, retries/duplicates/loss, write duration, CU, and resulting file sizes. "
                "Accept only if reliability and latency requirements remain satisfied while commit/file pressure and CU materially improve; include feature-flag rollback and replay instructions."
            )
        if not test_plan:
            test_plan.append(
                "CONTROL TEST | No change is authorized. Retain this table as a healthy control. Re-test only after a workload, write-pattern, or data-shape regression is observed; record the same correctness, duration, files/bytes, and CU metrics used for comparable benchmark tables."
            )

        remedy_sql = "\n".join(remedy_commands) if remedy_commands else "-- No action needed (Table is healthy or requires observation only)" 

        return {
            "lakehouse_name": lh_name,
            "lakehouse_id": lh_id,
            "table_fqn": fqn,
            "sql_ref": quoted_fqn,
            "schema_name": schema_name,
            "table_name": table_name,
            "layer": layer,
            "format": "delta",
            "is_shortcut": bool(is_shortcut),
            "shortcut_type": shortcut_type,
            "shortcut_target": shortcut_target,
            "health_status": health_status,
            "evidence_tier": evidence_tier,
            "test_plan": " | ".join(test_plan),
            "adaptive_target_file_size": adaptive_target_file_size,
            "priority_score": int(priority_score),
            "size_mb": float(size_mb),
            "num_files": int(num_files),
            "avg_file_mb": float(avg_file_mb),
            "min_file_mb": float(min_file_mb),
            "median_file_mb": float(median_file_mb),
            "max_file_mb": float(max_file_mb),
            "metric_status": metric_status,
            "metric_verification_note": metric_verification_note,
            "consumer_num_files": int(consumer_num_files),
            "consumer_size_mb": float(consumer_size_mb),
            "owner_num_files": int(owner_num_files),
            "owner_size_mb": float(owner_size_mb),
            "partition_cols": ",".join(part_cols),
            "clustering_cols": ",".join(cluster_cols),
            "candidate_key_cols": ",".join(candidate_key_cols),
            "total_cols": int(total_cols),
            "dv_enabled": bool(dv_enabled),
            "vorder_enabled": vorder_prop,
            "optimize_write": opt_write_prop,
            "auto_compact": auto_compact_prop,
            "column_mapping": col_mapping,
            "log_retention": log_ret_prop,
            "deleted_file_retention": del_ret_prop,
            "min_reader_version": int(min_reader),
            "min_writer_version": int(min_writer),
            "write_commits_7d": int(write_commits_7d),
            "merge_delete_commits_7d": int(merge_delete_update_commits_7d),
            "small_append_commits": int(small_append_commits),
            "days_since_optimize": int(days_since_optimize),
            "days_since_vacuum": int(days_since_vacuum),
            "days_since_modified": int(days_since_modified),
            "needs_optimize": bool(needs_optimize),
            "needs_vacuum": bool(needs_vacuum),
            "needs_analyze": bool(needs_analyze),
            "flags": " | ".join(flags) if flags else "NONE",
            "recommendations": " ; ".join(recommendations) if recommendations else "Table layout and maintenance are healthy.",
            "remedy_sql": remedy_sql,
            "scan_duration_s": round(time.time() - t0, 2)
        }
    except Exception as e:
        log_event("table_profile_error", level="ERROR", table=fqn, error=str(e))
        return {
            "lakehouse_name": lh_name, "lakehouse_id": lh_id,
            "table_fqn": fqn, "sql_ref": quoted_fqn, "schema_name": schema_name, "table_name": table_name,
            "layer": layer, "format": "error", "is_shortcut": bool(is_shortcut),
            "shortcut_type": shortcut_type, "shortcut_target": shortcut_target,
            "health_status": "ERROR", "evidence_tier": "INFORMATIONAL", "test_plan": "Resolve access/profile error before benchmarking.", "adaptive_target_file_size": "unknown",
            "priority_score": -1, "size_mb": 0.0, "num_files": 0, "avg_file_mb": 0.0,
            "min_file_mb": 0.0, "median_file_mb": 0.0, "max_file_mb": 0.0,
            "metric_status": "ERROR", "metric_verification_note": "Profile failed before metric verification.",
            "consumer_num_files": consumer_num_files, "consumer_size_mb": consumer_size_mb,
            "owner_num_files": owner_num_files, "owner_size_mb": owner_size_mb,
            "partition_cols": "", "clustering_cols": "", "candidate_key_cols": "", "total_cols": 0,
            "dv_enabled": False, "vorder_enabled": "error", "optimize_write": "error",
            "auto_compact": "error", "column_mapping": "error", "log_retention": "",
            "deleted_file_retention": "", "min_reader_version": 0, "min_writer_version": 0,
            "write_commits_7d": 0, "merge_delete_commits_7d": 0, "small_append_commits": 0,
            "days_since_optimize": -1, "days_since_vacuum": -1, "days_since_modified": -1,
            "needs_optimize": False, "needs_vacuum": False, "needs_analyze": False,
            "flags": "SCAN_ERROR", "recommendations": str(e),
            "remedy_sql": "-- Fix table access or corrupted _delta_log",
            "scan_duration_s": round(time.time() - t0, 2)
        }
    finally:
        spark.sparkContext.setJobDescription(None)

# ------------------------------------------------------------------------------
# Execute Deep Table Profiling Across All Discovered Tables in Parallel
# ------------------------------------------------------------------------------
profile_start = time.time()
profiled_tables = []

with ThreadPoolExecutor(max_workers=disc_workers) as pool:
    prof_futures = [pool.submit(profile_single_table, tbl) for tbl in all_candidate_tables]
    for fut in as_completed(prof_futures):
        profiled_tables.append(fut.result())

# Sort tables so the highest-priority issues and largest tables appear at the top
profiled_tables.sort(key=lambda x: (x.get("priority_score", 0), x.get("size_mb", 0.0)), reverse=True)

log_event(
    "discovery_completed",
    lakehouses_scanned=len(WORKSPACE_LAKEHOUSES),
    tables_scanned=len(profiled_tables),
    shortcut_tables_count=sum(1 for t in profiled_tables if t.get("is_shortcut")),
    critical_count=sum(1 for t in profiled_tables if t.get("health_status") == "CRITICAL"),
    warning_count=sum(1 for t in profiled_tables if t.get("health_status") == "WARNING"),
    healthy_count=sum(1 for t in profiled_tables if t.get("health_status") == "HEALTHY"),
    tables_needing_optimize=sum(1 for t in profiled_tables if t.get("needs_optimize")),
    tables_needing_vacuum=sum(1 for t in profiled_tables if t.get("needs_vacuum")),
    duration_s=round(time.time() - profile_start, 2)
)


---
### Step 4: Unmanaged `Files/` Shortcuts, Orphaned Streaming Checkpoints & Stale Table Scanner
**What this cell does:**
1. **Inspects `Files/` Shortcuts at Their Target Locations:** Walks unmanaged shortcut folders under `Files/` across every Lakehouse to count files, calculate average file size (MB), and flag `SMALL_FILES_IN_SHORTCUT`.
2. **Scans for Orphaned Streaming Checkpoints:** Safely lists `Files/` first (using explicit `abfss://` URIs so OneLake never throws HTTP `400 BadRequest`) to check for `Files/_checkpoints` directories left behind by decommissioned Structured Streaming jobs.
3. **Displays Interactive Summary Grids** for all discovered Table Shortcuts and `Files/` Shortcuts.


In [ ]:
# ==============================================================================
# STEP 4: WORKSPACE-WIDE FILES/ SHORTCUTS, CHECKPOINTS & STALE DATA SCANNER
# ==============================================================================

# ------------------------------------------------------------------------------
# 4.1 Inspect Unmanaged Files/ Shortcuts Across All Scanned Lakehouses
# ------------------------------------------------------------------------------
files_shortcut_health = []
if inspect_files_shortcuts and files_shortcuts_list:
    try:
        import notebookutils
        for fsc in files_shortcuts_list:
            # Build explicit abfss:// URI to the shortcut inside the Lakehouse
            sc_uri = f"{fsc['abfs_root']}/{fsc['full_path'].lstrip('/')}"
            file_count = 0
            total_bytes = 0
            try:
                # Walk up to 2 directory levels inside the Files/ shortcut to measure file sizes
                entries = notebookutils.fs.ls(sc_uri)
                for e in entries:
                    if not e.isDir:
                        file_count += 1
                        total_bytes += int(e.size or 0)
                    else:
                        sub_entries = notebookutils.fs.ls(e.path)
                        for se in sub_entries:
                            if not se.isDir:
                                file_count += 1
                                total_bytes += int(se.size or 0)
                total_mb = round(total_bytes / (1024.0 * 1024.0), 2)
                avg_mb = round(total_mb / file_count, 2) if file_count > 0 else 0.0
                sc_flag = "SMALL_FILES_IN_SHORTCUT" if (file_count >= 50 and avg_mb < small_file_size_threshold_mb) else "OK"
                files_shortcut_health.append({
                    "Lakehouse": fsc["lakehouse_name"],
                    "Shortcut Path": fsc["full_path"],
                    "Target Type": fsc["target_type"],
                    "Files": file_count,
                    "Total (MB)": total_mb,
                    "Avg File (MB)": avg_mb,
                    "Status": sc_flag,
                    "Target Location": fsc["target_location"]
                })
            except Exception as ex:
                files_shortcut_health.append({
                    "Lakehouse": fsc["lakehouse_name"],
                    "Shortcut Path": fsc["full_path"],
                    "Target Type": fsc["target_type"],
                    "Files": -1,
                    "Total (MB)": 0.0,
                    "Avg File (MB)": 0.0,
                    "Status": f"ACCESS_ERROR: {ex}",
                    "Target Location": fsc["target_location"]
                })
    except Exception as e:
        log_event("files_shortcut_scan_warning", level="WARN", error=str(e))

# ------------------------------------------------------------------------------
# 4.2 Inspect Files/_checkpoints for Orphaned Streaming State Stores
# ------------------------------------------------------------------------------
checkpoint_inventory = []
if scan_orphan_checkpoints_path:
    try:
        import notebookutils
        ckpt_dir_name = scan_orphan_checkpoints_path.strip("/").split("/")[-1].lower()
        for lh in WORKSPACE_LAKEHOUSES:
            lh_abfs = lh["abfs_root"]
            try:
                # List parent 'Files' directory first rather than calling fs.exists() on a missing child path
                root_entries = notebookutils.fs.ls(f"{lh_abfs}/Files")
                has_ckpt_dir = any(e.isDir and e.name.strip("/").lower() == ckpt_dir_name for e in root_entries)
                if has_ckpt_dir:
                    ckpt_uri = f"{lh_abfs}/{scan_orphan_checkpoints_path.lstrip('/')}"
                    for entry in notebookutils.fs.ls(ckpt_uri):
                        checkpoint_inventory.append({
                            "lakehouse_name": lh["lakehouse_name"],
                            "path": entry.path,
                            "name": entry.name,
                            "is_dir": entry.isDir,
                            "size_bytes": entry.size
                        })
            except Exception:
                pass
        log_event("checkpoint_scan_completed", lakehouses_checked=len(WORKSPACE_LAKEHOUSES), checkpoints_found=len(checkpoint_inventory))
    except Exception as e:
        log_event("checkpoint_scan_skipped", level="INFO", reason=str(e))

# ------------------------------------------------------------------------------
# 4.3 Display Summary Banner & Interactive DataFrame for Discovered Shortcuts
# ------------------------------------------------------------------------------
shortcut_tables = [t for t in profiled_tables if t.get("is_shortcut")]
stale_candidates = [t for t in profiled_tables if "STALE_OR_TEMP_TABLE_CANDIDATE" in (t.get("flags") or "")]

ipy_display(HTML(f"""
<div style="font-family:'Segoe UI',sans-serif; background:#f8fafc; border:1px solid #cbd5e1; border-left:5px solid #0284c7; border-radius:6px; padding:12px 16px; margin:8px 0;">
  <h4 style="margin:0 0 8px 0; color:#0f172a;">🔗 Workspace-Wide OneLake Shortcuts & Stale Storage Summary ({len(WORKSPACE_LAKEHOUSES)} Lakehouses Scanned)</h4>
  <span style="margin-right:24px;"><b>Table Shortcuts:</b> {len(shortcut_tables)}</span>
  <span style="margin-right:24px;"><b>Files/ Shortcuts:</b> {len(files_shortcut_health)}</span>
  <span style="margin-right:24px;"><b>Stale/Temp Table Candidates:</b> {len(stale_candidates)}</span>
  <span><b>Streaming Checkpoints:</b> {len(checkpoint_inventory)}</span>
</div>
"""))

if shortcut_tables:
    sc_pdf = pd.DataFrame([{
        "Lakehouse": st["lakehouse_name"],
        "Table FQN": st["table_fqn"],
        "Shortcut Type": st["shortcut_type"],
        "Health": st["health_status"],
        "Score": st["priority_score"],
        "Size (MB)": st["size_mb"],
        "Files": st["num_files"],
        "Avg File (MB)": st["avg_file_mb"],
        "Target Location": st["shortcut_target"],
        "Flags": st["flags"]
    } for st in shortcut_tables])
    try:
        display(spark.createDataFrame(sc_pdf))
    except Exception:
        ipy_display(sc_pdf)

if files_shortcut_health:
    fsc_pdf = pd.DataFrame(files_shortcut_health)
    ipy_display(fsc_pdf)


---
### Step 4B: Complete Implementation Planner & Approval Gate

Every detected finding becomes an action-level backlog item containing evidence status, expected benefit, risk, execution type, owner, approval state, phase, sequence, prerequisites, commands/tools, validation, measurable success criteria, rollback, dependencies, estimated effort, maintenance window, concurrency controls, and table metrics.

`audit_only` creates the plan without mutation. Other modes require the exact `lakehouse.schema.table` value in `approved_plan_table_fqns_json`. Shortcut and external actions remain owner-side/manual.


In [ ]:
# ==============================================================================
# STEP 4B: COMPLETE IMPLEMENTATION-PLAN GENERATOR
# ==============================================================================

approved_plan_table_fqns = json.loads(approved_plan_table_fqns_json or "[]")
if not isinstance(approved_plan_table_fqns, list) or not all(isinstance(x, str) for x in approved_plan_table_fqns):
    raise ValueError("approved_plan_table_fqns_json must be a JSON array of exact table FQNs")
approved_plan_table_fqns = {x.strip().lower() for x in approved_plan_table_fqns if x.strip()}

COMMON = {
    "VALIDATE": "Capture 5 repeated baseline runs; compare correctness, duration, files/bytes read, CU, and write/maintenance cost.",
    "SUCCESS": "Correctness passes and the approved improvement threshold is met without an unacceptable regression.",
}

ACTION_CATALOG = {
    "SMALL_FILE_PROBLEM": ("Validate and compact small files", "CONDITIONAL_TEST", "LOW", "APPROVED_MANUAL", "1_SAFE_MAINTENANCE", "Data Engineering", "SMALL",
        "Reduce file-open/task overhead and improve write/read efficiency.",
        "Confirm active regression, single-writer ownership, baseline metrics, and retention.",
        "Test future Optimize Write on a clone; benchmark; run incremental OPTIMIZE only if accepted.",
        "No direct layout undo. Preserve a pre-change clone/version and unset Optimize Write for future writes if rejected."),
    "SINGLE_LARGE_FILE_NO_SKIPPING": ("Improve selective-skipping layout", "CONDITIONAL_TEST", "MEDIUM", "APPROVED_MANUAL", "2_VALIDATE_AND_IMPLEMENT", "Data Engineering", "MEDIUM",
        "Increase file-level skipping for selective predicates.",
        "Identify actual predicates, create an equal clone, and verify runtime compatibility.",
        "Compare adaptive sizing and tested clustering/layout choices on the clone before promotion.",
        "Keep the original table; cut consumers back to it if acceptance fails."),
    "HIVE_PARTITIONED_UNDER_1TB": ("Evaluate Hive partition migration", "CONDITIONAL_TEST", "HIGH", "MANUAL_ONLY", "3_ARCHITECTURE_CHANGE", "Data Owner + Data Engineering", "LARGE",
        "Reduce directory/file fragmentation while preserving predicate locality.",
        "Owner approval, query inventory, clone, consumer inventory, and cutover plan.",
        "CTAS a tested unpartitioned or Liquid-Clustered variant; benchmark and coordinate cutover.",
        "Retain the original table until acceptance; switch consumers back on rejection."),
    "OVER_PARTITIONED_SMALL_FILES": ("Remediate over-partitioning", "CONDITIONAL_TEST", "HIGH", "MANUAL_ONLY", "3_ARCHITECTURE_CHANGE", "Data Owner + Data Engineering", "LARGE",
        "Reduce file/task and directory explosion.",
        "Verify partition contracts, owner approval, baseline consumers, and a clone.",
        "Test a lower-cardinality layout or Liquid Clustering, then promote atomically.",
        "Preserve and restore the original table/path."),
    "UNCLUSTERED_LARGE_TABLE": ("Test Liquid Clustering on real predicates", "CONDITIONAL_TEST", "MEDIUM", "APPROVED_MANUAL", "2_VALIDATE_AND_IMPLEMENT", "Data Engineering", "MEDIUM",
        "Improve co-location and file skipping for actual filters/joins.",
        "Use query-history predicates, not naming conventions; create a clone.",
        "Confirm 1-4 keys, apply CLUSTER BY to the clone, OPTIMIZE, and benchmark reads/writes.",
        "Do not cut over the original. Drop/archive a failed clone; physical layout needs rewrite."),
    "MUTABLE_TABLE_WITHOUT_DV": ("Enable and validate Deletion Vectors", "VALIDATED_SYNTHETIC", "MEDIUM", "APPROVED_MANUAL", "2_VALIDATE_AND_IMPLEMENT", "Data Engineering", "SMALL",
        "Reduce UPDATE/DELETE/MERGE rewrite amplification.",
        "Clone plus representative mutation/read workload and compatibility review.",
        "Benchmark DV-off and DV-on clones for mutation, reader latency, bytes rewritten, and DV density.",
        "Do not assume property reversal removes protocols/files; restore from the approved clone/version."),
    "NEEDS_MAINTENANCE_OPTIMIZE": ("Run incremental OPTIMIZE", "VALIDATED_SAFETY", "LOW", "AUTO_SAFE_APPROVED", "1_SAFE_MAINTENANCE", "Data Engineering", "SMALL",
        "Compact eligible files and purge applicable deletion vectors.",
        "One writer, maintenance window, pre-version baseline, active capacity, and adequate retention.",
        "Run incremental or scoped OPTIMIZE, verify table correctness, metrics, and workload SLA.",
        "No direct layout undo. Quiesce and restore from an approved clone/version if required."),
    "VACUUM_HISTORY_NOT_OBSERVED": ("Investigate orphan storage before VACUUM", "CONDITIONAL_TEST", "MEDIUM", "APPROVED_MANUAL", "1_SAFE_MAINTENANCE", "Data Platform + Data Owner", "SMALL",
        "Reclaim proven orphan storage; no assumed query benefit.",
        "Measure orphan files/bytes, old readers, backup/DR, and at least 168-hour retention.",
        "Verify evidence, then run VACUUM RETAIN 168 HOURS and record reclaimed storage/correctness.",
        "VACUUM is irreversible for eligible files. No rollback; rely on backup and retention."),
    "LOG_STORE_SMALL_COMMIT_ABUSE": ("Refactor per-event writes", "VALIDATED_SYNTHETIC", "HIGH", "MANUAL_ONLY", "3_ARCHITECTURE_CHANGE", "Application + Data Engineering", "LARGE",
        "Reduce commit/file/log explosion and capacity churn.",
        "Application owner, durability/latency contract, and approved buffering/event design.",
        "Buffer/micro-batch or route events appropriately; test delivery semantics, then migrate the producer.",
        "Feature-flag the producer and route to the prior approved path while preserving replay data."),
    "VORDER_CONSUMER_REVIEW": ("Test V-Order for the actual consumer", "CONDITIONAL_TEST", "MEDIUM", "EXTERNAL_REQUIRED", "4_EXTERNAL_OR_PLATFORM_VALIDATION", "BI + Data Engineering", "MEDIUM",
        "Possible read/compression benefit for a confirmed consumer.",
        "Identify the consumer and construct equal semantic/read variants.",
        "Run cold/warm consumer, write, size, refresh, CU, and correctness tests.",
        "Unset property for future writes; existing V-Order files remain until rewritten."),
    "FILTER_COLUMNS_OUTSIDE_STATS": ("Tune data-skipping statistics", "CONDITIONAL_TEST", "LOW", "APPROVED_MANUAL", "2_VALIDATE_AND_IMPLEMENT", "Data Engineering", "SMALL",
        "Improve file skipping for actual predicates.",
        "Confirm real filter/join columns and avoid expensive payload columns.",
        "Set explicit stats columns on a clone, rewrite representative data, and compare reads/writes.",
        "Restore the prior property for future commits; rewrite may be needed for physical stats."),
    "WIDE_STRING_STATS_REVIEW": ("Review costly statistics columns", "MANUAL_REVIEW", "LOW", "MANUAL_ONLY", "2_VALIDATE_AND_IMPLEMENT", "Data Engineering", "SMALL",
        "Reduce statistics write overhead on rarely filtered payload columns.",
        "Verify query history and acceptable payload-filter behavior.",
        "Select only useful stats columns, test, and apply after approval.",
        "Restore the prior property."),
    "STALE_OR_TEMP_TABLE_CANDIDATE": ("Archive or remove confirmed stale data", "MANUAL_REVIEW", "HIGH", "OWNER_REQUIRED", "3_ARCHITECTURE_CHANGE", "Business/Data Owner", "MEDIUM",
        "Reclaim storage and namespace while reducing estate risk.",
        "Owner, lineage, legal/retention, backup/archive, and consumer sign-off.",
        "Archive, remove dependencies, then drop only after approval and verify lineage/capacity.",
        "No automatic rollback. Restore through the approved archive/recovery procedure."),
}


def plan_template(table, finding):
    if finding.startswith("SHORTCUT("):
        item = ("Execute remediation at shortcut owner", "EXTERNAL_REQUIRED", "MEDIUM", "OWNER_SIDE_ONLY",
                "4_EXTERNAL_OR_PLATFORM_VALIDATION", "Source Owner", "MEDIUM",
                "Avoid consumer/source dual-writer conflicts and drift.",
                "Resolve target owner, writer, schedule, and source-platform procedure.",
                "Send this plan to the source owner; implement and verify at source, then validate the consumer.",
                "Rollback only at source under the source owner procedure.")
    elif finding == "CROSS_CLOUD_SHORTCUT_EGRESS_RISK":
        item = ("Measure and control cross-cloud egress", "EXTERNAL_REQUIRED", "MEDIUM", "EXTERNAL_REQUIRED",
                "4_EXTERNAL_OR_PLATFORM_VALIDATION", "Cloud Platform + FinOps", "MEDIUM",
                "Avoid unexpected latency and egress cost.", "Billing, network telemetry, and representative traffic.",
                "Measure bytes/cost, compare replication/materialization, and select an approved route.",
                "Restore the prior route only after owner approval.")
    elif finding == "SHORTCUT_METRICS_UNVERIFIED":
        item = ("Verify shortcut metrics at the source", "EXTERNAL_REQUIRED", "MEDIUM", "OWNER_SIDE_ONLY",
                "0_BASELINE_AND_ACCESS", "Source Owner + Data Engineering", "SMALL",
                "Prevent recommendations from using stale or incomplete shortcut file metrics.",
                "Resolve the shortcut target and obtain direct owner-side access or trusted storage inventory.",
                "Run DESCRIBE DETAIL and an owner-path physical-file reconciliation; then update the audit evidence.",
                "No change is authorized; retain consumer metrics as unverified observation only.")
    elif finding == "SCAN_ERROR":
        item = ("Resolve scan/access failure", "MANUAL_REVIEW", "HIGH", "OWNER_REQUIRED",
                "0_BASELINE_AND_ACCESS", "Fabric Admin + Data Engineering", "SMALL",
                "Restore safe assessment before remediation.", "Permissions/path/Delta-log diagnosis.",
                "Fix access or corruption, rerun audit, and only then generate executable commands.",
                "Undo any permission change through normal governance; no table mutation should occur.")
    else:
        item = ACTION_CATALOG.get(finding, (
            f"Review {finding}", "MANUAL_REVIEW", "MEDIUM", "MANUAL_ONLY", "2_VALIDATE_AND_IMPLEMENT",
            "Data Engineering", "SMALL", "Resolve the detected finding.", "Owner and workload review.",
            "Confirm, clone, validate, approve, and implement through normal change control.",
            "Use the approved clone/version recovery procedure."))
    title, evidence, risk, execution, phase, owner, effort, benefit, prereq, steps, rollback = item
    key_cols = table.get("candidate_key_cols") or table.get("clustering_cols") or ""
    ref = table.get("sql_ref") or table.get("table_fqn")
    command = "-- Manual review required; no safe automatic command generated."
    if finding == "MUTABLE_TABLE_WITHOUT_DV":
        command = f"ALTER TABLE {ref} SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'true');"
    elif finding == "NEEDS_MAINTENANCE_OPTIMIZE":
        command = f"OPTIMIZE {ref};"
    elif finding == "VACUUM_HISTORY_NOT_OBSERVED":
        command = f"-- Measure orphans first; if approved: VACUUM {ref} RETAIN 168 HOURS;"
    elif finding == "FILTER_COLUMNS_OUTSIDE_STATS" and key_cols:
        command = f"ALTER TABLE {ref} SET TBLPROPERTIES ('delta.dataSkippingStatsColumns' = '{key_cols}');"
    elif finding in ("UNCLUSTERED_LARGE_TABLE", "HIVE_PARTITIONED_UNDER_1TB", "OVER_PARTITIONED_SMALL_FILES") and key_cols:
        command = f"-- Candidate after clone test: ALTER TABLE {ref} CLUSTER BY ({key_cols.replace(',', ', ')});"
    elif finding == "SMALL_FILE_PROBLEM":
        command = f"-- Validate first on clone; then if approved: OPTIMIZE {ref};"
    elif finding == "STALE_OR_TEMP_TABLE_CANDIDATE":
        command = f"-- After owner/lineage/retention/archive approval: DROP TABLE {table.get('table_fqn')};"
    elif finding == "VORDER_CONSUMER_REVIEW":
        command = "-- No automatic V-Order DDL. Run the real consumer A/B test first."
    elif finding.startswith("SHORTCUT(") or finding == "CROSS_CLOUD_SHORTCUT_EGRESS_RISK":
        command = f"-- Execute at owner/source: {table.get('shortcut_target') or '<resolve target>'}. Never through the shortcut consumer."
    return dict(title=title, evidence=evidence, risk=risk, execution=execution, phase=phase,
                owner=owner, effort=effort, benefit=benefit, prereq=prereq, steps=steps,
                validate=COMMON["VALIDATE"], success=COMMON["SUCCESS"], rollback=rollback,
                command=command, dependency="Baseline, ownership, validation, and approval evidence")



BENEFIT_CATALOG = {
    "SMALL_FILE_PROBLEM": ("PERFORMANCE_AND_COST", "Fewer small-file open/task operations; lower scheduler overhead and potentially lower read or write duration.", "Median read/write duration, files read, task count, and CU", "after representative A/B test"),
    "SINGLE_LARGE_FILE_NO_SKIPPING": ("SELECTIVE_READ_PERFORMANCE", "Better file-level data skipping for selective filters and joins.", "Files/bytes read, query duration, and CU", "after a representative selective-query test"),
    "HIVE_PARTITIONED_UNDER_1TB": ("LAYOUT_SIMPLIFICATION_AND_WRITE_EFFICIENCY", "Fewer directories and small files while preserving useful query locality.", "Files/directories, write duration, query duration, and bytes read", "after clone/cutover validation"),
    "OVER_PARTITIONED_SMALL_FILES": ("LAYOUT_SIMPLIFICATION_AND_WRITE_EFFICIENCY", "Lower file and directory explosion, simpler maintenance, and fewer tasks.", "File count, task count, write duration, and maintenance duration", "after clone/cutover validation"),
    "UNCLUSTERED_LARGE_TABLE": ("READ_PERFORMANCE", "Improved file skipping and co-location for actual filter and join patterns.", "Files/bytes read, selective-query duration, and CU", "after real-predicate testing"),
    "MUTABLE_TABLE_WITHOUT_DV": ("MUTATION_EFFICIENCY", "Lower data rewrite during UPDATE, DELETE, and MERGE operations.", "Mutation duration, bytes rewritten, read latency, and DV density", "after DV-off/DV-on testing"),
    "NEEDS_MAINTENANCE_OPTIMIZE": ("PHYSICAL_LAYOUT_AND_MAINTENANCE", "More efficient file layout and removal of eligible deletion-vector overhead.", "File count/size, read duration, maintenance duration, and CU", "after approved incremental maintenance"),
    "VACUUM_HISTORY_NOT_OBSERVED": ("STORAGE_RECLAMATION", "Potential reclamation of confirmed orphan storage without claiming a query-speed benefit.", "Orphan bytes, reclaimed bytes, duration, and read correctness", "after owner-approved retention review"),
    "LOG_STORE_SMALL_COMMIT_ABUSE": ("WRITE_AND_CAPACITY_EFFICIENCY", "Fewer commits, files, log operations, and unnecessary capacity use.", "Commits/files per second, write duration, end-to-end latency, and CU", "after producer redesign validation"),
    "VORDER_CONSUMER_REVIEW": ("CONSUMER_READ_EFFICIENCY", "Potentially better compression and read efficiency for the actual consumer.", "Cold/warm consumer latency, write duration, table size, and CU", "after consumer A/B testing"),
    "FILTER_COLUMNS_OUTSIDE_STATS": ("SELECTIVE_READ_PERFORMANCE", "More effective file skipping for confirmed filter/join columns.", "Files/bytes read, query duration, and write overhead", "after statistics-column testing"),
    "WIDE_STRING_STATS_REVIEW": ("WRITE_EFFICIENCY", "Potentially lower statistics-collection cost for rarely filtered large columns.", "Write duration, CU, log size, and selective-read regression", "after query-history review"),
    "STALE_OR_TEMP_TABLE_CANDIDATE": ("STORAGE_AND_GOVERNANCE", "Reclaimed storage and a smaller, easier-to-govern data estate.", "Storage reclaimed, active consumers, lineage, and retention evidence", "after owner and lineage approval"),
    "SHORTCUT(": ("OWNERSHIP_AND_SAFETY", "Avoid consumer/source write conflicts while applying the change at the authoritative source.", "Source consistency, consumer freshness, and absence of dual-writer activity", "after source-owner validation"),
    "CROSS_CLOUD_SHORTCUT_EGRESS_RISK": ("COST_AND_LATENCY", "Potentially lower transfer cost and improved cross-cloud read latency.", "Transferred bytes, egress cost, latency, and freshness", "after billing/network testing"),
    "SHORTCUT_METRICS_UNVERIFIED": ("EVIDENCE_QUALITY", "Prevents incorrect file-size recommendations based on stale or incomplete shortcut evidence.", "Consumer/owner file counts, bytes, snapshot freshness, and verification status", "after owner-side reconciliation"),
    "SCAN_ERROR": ("ASSESSMENT_RELIABILITY", "Restores reliable profiling before any change is recommended or executed.", "Successful profile completion and populated required metrics", "after access/path repair"),
}

def benefit_profile(finding):
    if finding.startswith("SHORTCUT("):
        return BENEFIT_CATALOG["SHORTCUT("]
    return BENEFIT_CATALOG.get(finding, ("RISK_AND_GOVERNANCE", "Resolve the detected issue with less operational risk and clearer evidence.", "Correctness, duration, files/bytes, write cost, storage, and CU", "after workload and owner validation"))

def operational_disposition(action):
    execution = action["execution"]
    if execution == "NO_ACTION":
        return "MONITOR_ONLY"
    if execution in ("OWNER_SIDE_ONLY", "OWNER_REQUIRED", "EXTERNAL_REQUIRED"):
        return "OWNER_OR_EXTERNAL_REVIEW"
    if execution == "MANUAL_ONLY":
        return "DESIGN_OR_CHANGE_REVIEW"
    if execution == "AUTO_SAFE_APPROVED":
        return "PILOT_THEN_SCHEDULE"
    if execution == "APPROVED_MANUAL":
        return "PILOT_REQUIRED"
    return "REVIEW_REQUIRED"


VALIDATION_CATALOG = {    "HEALTHY_CONTROL": {"objective": "Retain a no-change baseline and detect future regression.", "variants": ["Current healthy state only"], "queries": ["No new benchmark required; reuse the representative workload on the next material change"], "repetitions": "No repeated test for this run. Re-run the assessment after a material schema, write-pattern, data-size, or consumer change.", "metrics": ["file count", "table size", "average file size", "assessment date", "workload/data-shape change trigger"], "acceptance": "No action is accepted or required; the row remains a healthy control.", "rollback": "No mutation and therefore no rollback.", "code_template": "Run the generated read-only DESCRIBE DETAIL cell only; do not OPTIMIZE, VACUUM, or rewrite the table."},
    "WIDE_STRING_STATS_REVIEW": {"objective": "Prove that broad string columns can be excluded from Delta statistics without reducing useful data skipping.", "variants": ["Current statistics columns", "Clone with confirmed filter-only statistics columns"], "queries": ["Recurring filtered query using every retained statistics column", "Write/commit representative batch", "Non-filtered full workload"], "repetitions": "One excluded warm-up plus 5 measured runs per query/variant; run one matched representative write batch.", "metrics": ["files/bytes read", "query duration", "write and commit duration", "Delta-log size", "correctness"], "acceptance": "Write overhead improves, retained filters still skip comparable data, full correctness matches, and no selective query regresses beyond the approved limit.", "rollback": "Restore the prior statistics property and keep the original table as the production path.", "code_template": "Record the current property, create an equal clone, set only query-proven statistics columns, replay the identical writes and queries, and compare all listed metrics."},
    "NONE": {"objective": "Retain a healthy control and detect regressions.", "variants": ["Healthy current state"], "queries": ["Existing representative workload baseline"], "repetitions": "No test until workload/data-shape regression; then use 5 measured repetitions after a warm-up.", "metrics": ["correctness", "file count/size", "duration", "files/bytes read", "CU"], "acceptance": "No regression; otherwise create a new finding-specific protocol.", "rollback": "No change is authorized.", "code_template": "Record baseline metrics and do not execute maintenance solely because the table is selected as a control."},
    "SHORTCUT(": {"objective": "Verify shortcut source metrics and consumer behavior before source-owner remediation.", "variants": ["Shortcut consumer view", "Direct owner/source view"], "queries": ["Owner DESCRIBE DETAIL", "Consumer DESCRIBE DETAIL", "Representative shortcut read"], "repetitions": "One reconciled metadata snapshot plus 5 measured consumer reads; repeat after any owner-side maintenance.", "metrics": ["owner/consumer file count and bytes", "snapshot freshness", "files/bytes read", "duration", "errors", "egress/cost"], "acceptance": "Owner and consumer metadata reconcile or differences are documented; no consumer-side maintenance occurs; correctness matches.", "rollback": "No consumer-side change; source owner follows its approved recovery plan.", "code_template": "Resolve target, run DESCRIBE DETAIL and physical metadata at owner and consumer, then benchmark read workload only through consumer."},
    "OVER_PARTITIONED_SMALL_FILES": {"objective": "Quantify excessive partition overhead and validate a lower-cardinality or clustered alternative.", "variants": ["Current over-partitioned table", "Lower-cardinality partition candidate", "Liquid-Clustered candidate"], "queries": ["Partition-aligned query", "Cross-partition query", "Full-table aggregate"], "repetitions": "Five measured query and write cycles per variant with identical data, compute, and cache protocol.", "metrics": ["partition count", "file count per partition", "average file MB", "task count", "write/read duration", "bytes read", "CU"], "acceptance": "Candidate reduces fragmentation/task overhead without breaking write isolation, selective reads, full scans, or correctness.", "rollback": "Do not replace original until consumers validate; restore original partition path on failure.", "code_template": "Build equal clones, apply candidate layouts, replay fixed batched writes and queries, and compare partition/file/task metrics."},
    "HIVE_PARTITIONED_UNDER_1TB": {"objective": "Determine whether partitioning is justified by concurrent-write isolation or a simpler clustered layout is preferable.", "variants": ["Current Hive-partitioned table", "Equal unpartitioned clone", "Optional Liquid-Clustered clone"], "queries": ["Partition-aligned query", "Multi-predicate query", "Full-table aggregate"], "repetitions": "Five measured executions per query/variant after one excluded warm-up; separately run fresh-session cold tests.", "metrics": ["files/directories", "bytes read", "duration", "write conflicts", "small files", "maintenance duration", "CU"], "acceptance": "Chosen layout preserves concurrent-write requirements and improves the read/write/maintenance balance.", "rollback": "Keep the original until consumers and writers validate; cut back to the original on failure.", "code_template": "Create equal clones; replay identical writes and queries; compare directories, files, bytes, tasks, conflicts, duration, and correctness."},
    "NEEDS_MAINTENANCE_OPTIMIZE": {"objective": "Validate that compaction is needed and that incremental OPTIMIZE improves a representative workload.", "variants": ["Current pre-OPTIMIZE snapshot", "Post-incremental-OPTIMIZE snapshot"], "queries": ["Representative selective query", "Representative full-table aggregate", "Representative write/merge if applicable"], "repetitions": "Capture a baseline with one excluded warm-up plus 5 measured runs; run one controlled OPTIMIZE; repeat the same queries 5 times.", "metrics": ["correctness checksum", "min/median/max duration", "files/bytes read", "file count/size", "OPTIMIZE duration", "Capacity Units"], "acceptance": "Correctness matches and query or approved physical-layout improvement clears the threshold without excessive write, maintenance, or CU cost.", "rollback": "Keep the original/clone recovery path; layout rewrites are not directly reversible.", "code_template": "Run identical forced-action queries before and after one controlled OPTIMIZE and compare medians, files, bytes, and checksums."},

    "SMALL_FILE_PROBLEM": {
        "objective": "Prove that Optimize Write or one controlled OPTIMIZE improves representative workload cost.",
        "variants": ["Current/no-change control", "Optimize Write enabled on an equal write path", "Controlled incremental OPTIMIZE copy"],
        "queries": ["Representative selective filter/aggregation", "Representative full-table aggregate"],
        "repetitions": "5 measured repetitions per query and variant after 1 excluded warm-up; report cold-session runs separately.",
        "metrics": ["correctness checksum", "min/median/max duration", "files read", "bytes read", "Capacity Units", "file count/size", "write or OPTIMIZE duration"],
        "acceptance": "Correctness matches and approved median benefit is achieved without exceeding the write, maintenance, or CU regression limit.",
        "rollback": "Keep the original table; disable prospective Optimize Write and do not promote the tested copy when criteria fail.",
        "code_template": "Use a shared benchmark function: execute the exact Spark SQL/DataFrame twice as warm/cold protocol, call .collect() or .count() to force work, time with time.perf_counter(), and read Spark SQL metrics/files/bytes from the completed query.",
    },
    "UNCLUSTERED_LARGE_TABLE": {
        "objective": "Validate clustering keys from real filter/join behavior.",
        "variants": ["Current unclustered control", "Liquid-Clustered clone using observed keys"],
        "queries": ["Equality/range selective query", "Multi-predicate query", "Full-table aggregate"],
        "repetitions": "1 excluded warm-up plus 5 measured repetitions per query/variant; optional fresh-session cold repeat.",
        "metrics": ["correctness checksum", "files/bytes read", "files skipped", "duration min/median/max", "CU", "OPTIMIZE duration", "file distribution"],
        "acceptance": "Selective workloads meet the benefit threshold; full scan and maintenance regressions remain within approved limits; keys map to recurring real predicates.",
        "rollback": "Retain original table; drop/archive failed clone and do not cut consumers over.",
        "code_template": "CREATE TABLE clone CLUSTER BY (<observed keys>) AS SELECT * FROM control; OPTIMIZE clone; benchmark identical SQL against both references.",
    },
    "MUTABLE_TABLE_WITHOUT_DV": {
        "objective": "Measure Deletion Vector mutation/read trade-off.",
        "variants": ["DV-off clone", "DV-on clone"],
        "queries": ["Representative UPDATE/DELETE/MERGE", "Post-mutation read"],
        "repetitions": "Five measured operation cycles using fresh/reset clones; identical source and predicate.",
        "metrics": ["mutation duration", "files/bytes rewritten", "DV density", "post-mutation read files/bytes/duration", "CU", "correctness checksum"],
        "acceptance": "Mutation improvement exceeds threshold while post-mutation reads, cleanup cost, and correctness remain acceptable.",
        "rollback": "Keep DV-off original; restore/reset clone and do not change target property when criteria fail.",
        "code_template": "Clone source table; ALTER clone SET TBLPROPERTIES ('delta.enableDeletionVectors'='true'); replay identical mutation and read workload on both clones.",
    },
    "FILTER_COLUMNS_OUTSIDE_STATS": {
        "objective": "Validate explicit file-statistics columns.",
        "variants": ["Current statistics coverage", "Clone with explicit workload columns"],
        "queries": ["Equality filter", "Range filter", "Non-target control filter"],
        "repetitions": "1 excluded warm-up plus 5 measured repetitions per query and variant.",
        "metrics": ["files/bytes read", "duration", "CU", "write/commit duration", "Delta-log overhead", "correctness"],
        "acceptance": "Selective reads improve without unacceptable write overhead.",
        "rollback": "Restore prior statistics property for future writes and retain original table.",
        "code_template": "ALTER clone SET TBLPROPERTIES ('delta.dataSkippingStatsColumns'='<observed filter columns>'); rewrite/optimize clone as required; run identical filtered queries.",
    },
    "VORDER_CONSUMER_REVIEW": {
        "objective": "Measure V-Order for the actual consumer.",
        "variants": ["V-Order off", "V-Order on"],
        "queries": ["Actual Spark query set", "Actual Direct Lake/semantic-model queries when applicable"],
        "repetitions": "Five warm repetitions plus separate fresh-session cold runs; do not combine cold and warm medians.",
        "metrics": ["read/refresh duration", "write duration", "table size", "files/bytes read", "CU", "correctness"],
        "acceptance": "Confirmed consumer improves by approved threshold and write/refresh cost stays within limits.",
        "rollback": "Do not promote candidate; unset future-write property and retain original.",
        "code_template": "Write equal tables with parquet.vorder.enabled=false/true or table property variants; benchmark the actual consumer and record Spark/semantic-model metrics.",
    },
    "VACUUM_HISTORY_NOT_OBSERVED": {
        "objective": "Prove orphan storage reclamation without risking active data.",
        "variants": ["Pre-cleanup clone", "Post-VACUUM clone"],
        "queries": ["Current-state row/checksum verification", "Active-reader query", "Physical orphan inventory"],
        "repetitions": "One controlled cleanup plus at least 3 correctness/read checks before and after.",
        "metrics": ["orphan bytes", "reclaimed bytes", "VACUUM duration", "current file count", "row/checksum correctness", "active-reader success"],
        "acceptance": "Eligible orphan storage is materially reclaimed while current data, time-travel policy, and readers remain valid.",
        "rollback": "VACUUM is irreversible; rely on clone/backup and do not execute on the original until approved.",
        "code_template": "Use a disposable clone. Inventory files, record correctness, run VACUUM clone RETAIN 168 HOURS, then recheck correctness and file inventory.",
    },
    "LOG_STORE_SMALL_COMMIT_ABUSE": {
        "objective": "Validate a lower-commit write architecture.",
        "variants": ["Current per-event writes", "Buffered batch/micro-batch"],
        "queries": ["Write/replay workload", "Downstream read/delivery validation"],
        "repetitions": "Five fixed-volume event runs per variant.",
        "metrics": ["commits/files", "events per second", "end-to-end latency", "duplicates/loss/retries", "write duration", "CU", "correctness"],
        "acceptance": "Durability and latency remain correct while commit/file pressure and CU improve materially.",
        "rollback": "Feature flag producer; route to prior path and replay retained events.",
        "code_template": "Write a fixed event dataset through current and buffered paths; compare Delta history commit/file metrics and downstream result equality.",
    },
}

def validation_spec(finding):
    if finding.startswith("SHORTCUT("):
        return VALIDATION_CATALOG["SHORTCUT("]
    return VALIDATION_CATALOG.get(finding, {
        "objective": "Resolve the detected finding with reproducible evidence.",
        "variants": ["Control/current", "Approved candidate"],
        "queries": ["Representative selective workload", "Representative full workload"],
        "repetitions": "One excluded warm-up plus at least 5 measured repetitions; report cold runs separately.",
        "metrics": ["correctness", "duration", "files/bytes read", "CU", "file count/size"],
        "acceptance": "Correctness passes and the approved benefit/rollback limits are met.",
        "rollback": "Retain original and rollback using the approved clone/version or source-owner procedure.",
        "code_template": "Execute the exact same workload against control and candidate; force an action, time it, and compare metrics.",
    })


BEGINNER_GLOSSARY = (
    "Clone = a separate test copy; source = the original production table; "
    "warm-up = one unmeasured run used to start caches; median = the middle of five measured durations; "
    "CU (Capacity Units) = Fabric consumption shown in Monitoring Hub/Capacity Metrics; "
    "orphan file = a physical data file no longer referenced by the current Delta transaction log; "
    "Liquid Clustering = Delta's CLUSTER BY layout for recurring filter/join patterns; "
    "quiesce writers = pause or coordinate processes that write to the table."
)

END_USER_CATALOG = {
    "HEALTHY_CONTROL": {
        "meaning": "No current change is justified. This table is a healthy comparison point, not a maintenance task.",
        "decision": "Do nothing now. Re-run the assessment only after the data shape or workload materially changes.",
    },
    "SMALL_FILE_PROBLEM": {
        "meaning": "The table has many small data files. Spark may need more file-open and task-scheduling work than necessary.",
        "decision": "Change nothing until the same real queries show a benefit on a clone. File count alone is not proof that production needs maintenance.",
    },
    "NEEDS_MAINTENANCE_OPTIMIZE": {
        "meaning": "The table layout has enough fragmentation or eligible cleanup work that a controlled OPTIMIZE may help.",
        "decision": "Run OPTIMIZE only after a baseline, exact-table approval, and a confirmed writer-free maintenance window. Do not treat this as automatic permission.",
    },
    "VACUUM_HISTORY_NOT_OBSERVED": {
        "meaning": "The audit did not find enough evidence to prove regular VACUUM history. It does not prove orphan files exist.",
        "decision": "Do not run VACUUM just because history is missing. First prove old unreferenced files exist and confirm that backups, long readers, and time travel still need them.",
    },
    "UNCLUSTERED_LARGE_TABLE": {
        "meaning": "A large table is not Liquid Clustered, but Spark has not yet proven that useful recurring filter/join columns exist.",
        "decision": "Do not cluster by guessed column names. Get real query predicates, build an equal test copy, and cut over only if selective reads improve without unacceptable full-scan, write, and maintenance cost.",
    },
    "HIVE_PARTITIONED_UNDER_1TB": {
        "meaning": "The table uses Hive-style directory partitions below the framework's size heuristic. The layout may create complexity, but it can still be valid for concurrent writers.",
        "decision": "Keep the current layout until you prove that partitioning is not required for writer isolation or recurring queries. Compare unpartitioned and/or clustered test copies before migration.",
    },
    "OVER_PARTITIONED_SMALL_FILES": {
        "meaning": "Too many partitions may be creating many tiny files and more Spark tasks than the workload needs.",
        "decision": "Do not delete partition columns immediately. Verify downstream partition contracts, then test a lower-cardinality or clustered copy with the same writes and reads.",
    },
    "MUTABLE_TABLE_WITHOUT_DV": {
        "meaning": "UPDATE, DELETE, or MERGE may rewrite more Parquet data because Deletion Vectors are not enabled.",
        "decision": "Test Deletion Vectors on an isolated copy with the real mutation and post-mutation read workload; do not enable them on production based only on this flag.",
    },
    "FILTER_COLUMNS_OUTSIDE_STATS": {
        "meaning": "Recurring filter or join columns may not be covered by Delta's collected data-skipping statistics.",
        "decision": "Use query evidence, not column-name guesses. Add only useful columns on a test copy and measure both read improvement and write overhead.",
    },
    "WIDE_STRING_STATS_REVIEW": {
        "meaning": "Large string columns may be expensive to store in Delta statistics when they are not useful filters.",
        "decision": "Do not remove statistics until query history proves those columns are rarely used in WHERE/JOIN filters and selective reads remain safe.",
    },
    "VORDER_CONSUMER_REVIEW": {
        "meaning": "V-Order may help compression/read performance for a specific consumer, but the table's layer name cannot prove it.",
        "decision": "Test identical V-Order-off and V-Order-on copies with the actual Spark/Power BI workload. Keep it only when the real consumer improves within the approved write/refresh cost limit.",
    },
    "LOG_STORE_SMALL_COMMIT_ABUSE": {
        "meaning": "The producer is committing very small batches too often, which can grow Delta history and file/task overhead.",
        "decision": "This is an application/streaming design change. Preserve delivery guarantees, test fixed event volumes, and migrate the producer behind a rollback switch.",
    },
    "SHORTCUT(": {
        "meaning": "This table is a shortcut to data owned elsewhere. The consumer is a reader, not the authoritative maintenance owner.",
        "decision": "Never OPTIMIZE or VACUUM through the shortcut. Send the target and evidence to the source owner, validate at source, then rerun consumer-side read checks.",
    },
    "SHORTCUT_METRICS_UNVERIFIED": {
        "meaning": "The shortcut's source file metrics could not be reconciled with enough confidence for a file-size decision.",
        "decision": "Obtain owner-side evidence. No size-based remediation is authorized from incomplete consumer metrics.",
    },
    "CROSS_CLOUD_SHORTCUT_EGRESS_RISK": {
        "meaning": "Reading this shortcut may transfer data across cloud/network boundaries and create avoidable cost or latency.",
        "decision": "Measure transferred bytes, latency, and actual billing/network cost before changing the route.",
    },
    "SCAN_ERROR": {
        "meaning": "The assessment could not read enough metadata to make a reliable recommendation.",
        "decision": "Fix access or table/path health first. Do not execute maintenance or optimization for an incomplete scan.",
    },
}


def end_user_profile(finding):
    if finding.startswith("SHORTCUT("):
        return END_USER_CATALOG["SHORTCUT("]
    return END_USER_CATALOG.get(finding, {
        "meaning": "The audit found a condition that needs owner and workload evidence before action.",
        "decision": "Do not execute a change until the prerequisite, real-workload test, approval, and rollback conditions are satisfied.",
    })


def validation_starter_cell(finding, table):
    ref = table.get("sql_ref") or table.get("table_fqn") or "<table>"
    if finding == "HEALTHY_CONTROL":
        return f'''# Read-only health check; no change is recommended.
SOURCE = {ref!r}
detail = spark.sql(f"DESCRIBE DETAIL {{SOURCE}}").select("name", "numFiles", "sizeInBytes").collect()[0].asDict()
print(detail)
print("No maintenance action: rerun after a material workload or data-shape change.")'''
    if finding.startswith("SHORTCUT("):
        target = table.get("shortcut_target") or "<source-owner-target>"
        return f'''# Consumer-side read checks only. Never OPTIMIZE/VACUUM a shortcut here.
CONSUMER = {ref!r}
OWNER_TARGET = {target!r}
print("Consumer metadata:")
display(spark.sql(f"DESCRIBE DETAIL {{CONSUMER}}"))
print("Send the owner target and evidence to the source owner before any remediation:", OWNER_TARGET)
# Replace the query below with an aggregate query actually used by a consumer.
CONSUMER_QUERY = f"SELECT COUNT(1) AS rows_read FROM {{CONSUMER}}"
display(spark.sql(CONSUMER_QUERY))'''
    if finding in ("VACUUM_HISTORY_NOT_OBSERVED", "SHORTCUT_METRICS_UNVERIFIED", "CROSS_CLOUD_SHORTCUT_EGRESS_RISK", "SCAN_ERROR"):
        profile = end_user_profile(finding)
        return f'''# Evidence check only; this cell performs no optimization or VACUUM.
SOURCE = {ref!r}
print({profile["meaning"]!r})
display(spark.sql(f"DESCRIBE DETAIL {{SOURCE}}"))
print("STOP here until the owner, evidence, retention, and access checks in decision_rule are complete.")'''
    if finding in ("LOG_STORE_SMALL_COMMIT_ABUSE", "VORDER_CONSUMER_REVIEW"):
        profile = end_user_profile(finding)
        return f'''# Notebook-side starter only; the real producer/consumer must be tested.
SOURCE = {ref!r}
print({profile["meaning"]!r})
display(spark.sql(f"DESCRIBE DETAIL {{SOURCE}}"))
# Replace this with the application's fixed-volume replay or the actual consumer's query/refresh.
REAL_WORKLOAD_QUERY = "<PASTE_REAL_CONSUMER_QUERY>"
if REAL_WORKLOAD_QUERY.startswith("<"):
    raise ValueError("Replace REAL_WORKLOAD_QUERY with the production query before testing.")
display(spark.sql(REAL_WORKLOAD_QUERY))
print("Record run duration, output, and Fabric CU from Monitoring Hub/Capacity Metrics.")'''
    return f'''# Safe benchmark starter. Paste real production aggregate queries before running.
from time import perf_counter
from statistics import median
SOURCE = {ref!r}
# Each query must return a small aggregate result; do not use SELECT * on a large table.
SELECTIVE_QUERY = f"SELECT COUNT(1) AS selected_rows FROM {{SOURCE}} WHERE <PASTE_REAL_FILTER>"
FULL_QUERY = "<PASTE_REAL_FULL_TABLE_AGGREGATE>"
if "<PASTE" in SELECTIVE_QUERY or FULL_QUERY.startswith("<"):
    raise ValueError("Paste a real selective query and a real full-table aggregate before benchmarking.")

def benchmark(label, query, measured_runs=5):
    spark.sql(query).collect()  # warm-up; intentionally not measured
    seconds = []
    expected = None
    for _ in range(measured_runs):
        started = perf_counter()
        actual = spark.sql(query).collect()
        seconds.append(perf_counter() - started)
        if expected is None:
            expected = actual
        if actual != expected:
            raise AssertionError(f"{{label}} returned inconsistent results")
    return {{"label": label, "seconds": seconds, "median_seconds": median(seconds), "result": expected}}

results = [benchmark("selective", SELECTIVE_QUERY), benchmark("full", FULL_QUERY)]
detail = spark.sql(f"DESCRIBE DETAIL {{SOURCE}}").select("numFiles", "sizeInBytes").first().asDict()
print({{"detail": detail, "benchmarks": results}})
print("Save these results; repeat the exact same queries and run settings on each approved test variant.")'''


def action_brief(finding, table, action):
    ref = table.get("sql_ref") or table.get("table_fqn") or "<table>"
    briefs = {
        "HEALTHY_CONTROL": {
            "tool": "This notebook or a read-only Fabric notebook cell",
            "summary": "No remediation is needed now; keep this table as a healthy baseline.",
            "steps": "1) Record current row/file/size metadata. 2) Do not run maintenance. 3) Save the representative workload details. 4) Re-audit after a material data or workload change."
        },
        "SHORTCUT(": {
            "tool": "Source-owner Fabric or ADLS management tools; consumer-side Fabric notebook for read validation only",
            "summary": "Leave the shortcut unchanged and send the exact source target plus evidence to the source owner.",
            "steps": "1) Copy the owner target from this row. 2) Send the finding and DESCRIBE DETAIL evidence to the owner. 3) Owner validates and remediates at source. 4) Consumer reruns the same read query and compares results, duration, files/bytes read, and CU. 5) Never run OPTIMIZE or VACUUM through the shortcut."
        },
        "NEEDS_MAINTENANCE_OPTIMIZE": {
            "tool": "This notebook (approved optimize mode) or Fabric Spark SQL",
            "summary": f"Run one controlled incremental OPTIMIZE on {table.get('table_fqn')} and verify the same queries before and after.",
            "steps": f"1) Capture correctness and query baseline. 2) Quiesce or coordinate writers. 3) Run OPTIMIZE {ref}. 4) Record file count/MB, duration, files/bytes read, and CU. 5) Repeat audit and benchmark."
        },
        "SMALL_FILE_PROBLEM": {
            "tool": "Notebook validation copy + Fabric Spark SQL/PySpark",
            "summary": "Validate the workload first; if measured beneficial, enable Optimize Write for future writes and run one controlled incremental OPTIMIZE.",
            "steps": "1) Identify a real selective and full query. 2) Benchmark current state. 3) test Optimize Write on an equal write path. 4) test incremental OPTIMIZE on a copy. 5) approve only the variant that clears the threshold."
        },
        "UNCLUSTERED_LARGE_TABLE": {
            "tool": "Fabric Spark SQL / notebook CTAS + OPTIMIZE",
            "summary": "Build a Liquid-Clustered copy using columns observed in real filters/joins, then benchmark before cutover.",
            "steps": "1) Extract real filter/join columns. 2) Create equal clustered clone. 3) run OPTIMIZE. 4) benchmark identical queries. 5) cut over only after acceptance."
        },
        "HIVE_PARTITIONED_UNDER_1TB": {
            "tool": "Fabric Spark SQL / CTAS in a controlled notebook",
            "summary": "Assess whether Hive partitioning is justified by concurrent writes; otherwise test an unpartitioned or Liquid-Clustered replacement.",
            "steps": "1) Inventory partition and concurrent-writer needs. 2) build equal replacement clone. 3) replay writes and queries. 4) compare directories/files/bytes/CU. 5) owner-approved cutover."
        },
        "OVER_PARTITIONED_SMALL_FILES": {
            "tool": "Fabric Spark SQL / CTAS + workload replay notebook",
            "summary": "Replace excessive partitioning with a tested lower-cardinality or Liquid-Clustered layout.",
            "steps": "1) Confirm partition contracts. 2) create candidate clone. 3) replay representative writes. 4) compare file/task/query metrics. 5) cut over with rollback table retained."
        },
        "MUTABLE_TABLE_WITHOUT_DV": {
            "tool": "Fabric Spark SQL on an isolated clone",
            "summary": "Test Deletion Vectors on an equal clone before changing the source table.",
            "steps": "1) clone the table. 2) ALTER clone table property to enable deletion vectors. 3) replay same mutation. 4) compare rewritten bytes, duration, read latency, and correctness. 5) approve or reject."
        },
        "VACUUM_HISTORY_NOT_OBSERVED": {
            "tool": "Notebook/OneLake file inventory first; VACUUM SQL only after evidence",
            "summary": "Do not run from history alone. Inventory orphan files first and run VACUUM only when aged, unreferenced storage is proven.",
            "steps": f"1) Inventory unreferenced files/bytes and age. 2) verify readers, time travel, and backup. 3) if approved, run VACUUM {ref} RETAIN 168 HOURS. 4) verify correctness and reclaimed bytes."
        },
        "FILTER_COLUMNS_OUTSIDE_STATS": {
            "tool": "Fabric Spark SQL on an equal clone",
            "summary": "Add only confirmed filter/join columns to explicit data-skipping statistics, then test read and write cost.",
            "steps": "1) confirm columns from real queries. 2) alter clone statistics property. 3) rewrite/optimize clone. 4) run identical filters. 5) compare files/bytes, duration, write cost, and CU."
        },
        "WIDE_STRING_STATS_REVIEW": {
            "tool": "Query-history review + Fabric Spark SQL on clone",
            "summary": "Confirm the wide columns are not useful filters; test reduced statistics coverage to lower write overhead without harming reads.",
            "steps": "1) inspect recurring query predicates. 2) choose useful stats columns. 3) test clone. 4) compare write duration/log overhead and read regressions. 5) approve only if net-positive."
        },
        "VORDER_CONSUMER_REVIEW": {
            "tool": "Fabric notebook plus actual Power BI/Spark consumer test",
            "summary": "Test V-Order against the confirmed consumer; do not infer it from table/layer names.",
            "steps": "1) create off/on equal tables. 2) bind/run the actual consumer. 3) capture cold/warm read, write, refresh, size, CU. 4) keep V-Order only if consumer benefit clears the threshold."
        },
        "LOG_STORE_SMALL_COMMIT_ABUSE": {
            "tool": "Application/streaming redesign + Fabric notebook batch replay",
            "summary": "Replace per-event Delta writes with an approved buffered or micro-batch design.",
            "steps": "1) inventory producer. 2) define durability/latency contract. 3) implement buffered/streaming candidate. 4) replay fixed event volume. 5) compare commits/files/CU and delivery correctness."
        },
        "STALE_OR_TEMP_TABLE_CANDIDATE": {
            "tool": "Owner/lineage review, archive tooling, then SQL DROP after approval",
            "summary": "Confirm ownership and dependencies, archive or retain approved data, then remove the stale object.",
            "steps": "1) owner and lineage review. 2) retention/legal check. 3) archive/backup. 4) disconnect consumers. 5) approved DROP and post-check."
        },
        "VACUUM_HISTORY_NOT_OBSERVED": {
            "tool": "Notebook/OneLake physical file inventory; then VACUUM only if evidence exists",
            "summary": "Treat missing VACUUM history as informational. Investigate orphan bytes and execute only after the retention and reader review passes.",
            "steps": "1) inventory unreferenced aged files. 2) verify active readers/time travel. 3) run clone proof. 4) execute approved source VACUUM at 168 hours. 5) verify correctness."
        },
    }
    if finding.startswith("SHORTCUT(") or finding in ("SHORTCUT_METRICS_UNVERIFIED", "CROSS_CLOUD_SHORTCUT_EGRESS_RISK"):
        key = finding if finding in briefs else "SHORTCUT("
    else:
        key = finding
    result = briefs.get(key)
    if result:
        return result
    return {
        "tool": "Owner review plus the command/tool shown in this action",
        "summary": action.get("title", "Review the detected condition before changing it."),
        "steps": action.get("steps", "1) Establish baseline. 2) validate on a clone/control. 3) approve. 4) execute with the listed command. 5) verify and retain rollback.")
    }

implementation_plan_rows = []
for table in profiled_tables:
    fqn = table.get("table_fqn", "")
    findings = [x for x in (table.get("flags") or "NONE").split(" | ") if x]
    if findings == ["NONE"]:
        findings = ["HEALTHY_CONTROL"]
    approved = fqn.lower() in approved_plan_table_fqns
    for sequence, finding in enumerate(findings, start=1):
        if finding == "HEALTHY_CONTROL":
            action = dict(
                title="Monitor as healthy control", evidence="HEALTHY_CONTROL", risk="NONE",
                execution="NO_ACTION", phase="0_BASELINE_AND_ACCESS", owner="Data Engineering",
                effort="NONE", benefit="Preserve a comparison baseline and avoid unnecessary change.",
                prereq="None.", steps="No implementation; re-audit after material workload/data-shape change.",
                validate=table.get("test_plan") or "Monitor for workload regression.",
                success="Health remains stable.", rollback="Not applicable.", command="-- No action.",
                dependency="None")
        else:
            action = plan_template(table, finding)
        if table.get("is_shortcut") and action["execution"] not in ("OWNER_SIDE_ONLY", "EXTERNAL_REQUIRED", "NO_ACTION"):
            action.update(
                execution="OWNER_SIDE_ONLY",
                owner="Source Owner + Consumer Owner",
                command=f"-- Apply at source owner: {table.get('shortcut_target')}. Never execute through consumer shortcut.",
                rollback="Rollback only at the source under the source owner procedure.")
        if action["execution"] in ("APPROVED_MANUAL", "AUTO_SAFE_APPROVED"):
            approval = "APPROVED_FOR_THIS_RUN" if approved else "PENDING_EXPLICIT_APPROVAL"
        elif action["execution"] in ("MANUAL_ONLY", "OWNER_REQUIRED", "OWNER_SIDE_ONLY", "EXTERNAL_REQUIRED"):
            approval = "REQUIRED_EXTERNAL_OR_OWNER"
        else:
            approval = "NOT_REQUIRED"
        implementation_plan_rows.append({
            "run_id": RUN_ID, "run_timestamp_utc": RUN_TIMESTAMP_UTC,
            "plan_id": f"{RUN_ID}-{len(implementation_plan_rows) + 1:04d}",
            "lakehouse_name": table.get("lakehouse_name", ""), "lakehouse_id": table.get("lakehouse_id", ""),
            "table_fqn": fqn, "table_type": "Shortcut" if table.get("is_shortcut") else "Native",
            "finding": finding, "action_title": action["title"], "evidence_status": action["evidence"],
            "risk": action["risk"], "execution_type": action["execution"], "owner": action["owner"],
            "approval_status": approval, "implementation_phase": action["phase"], "sequence": sequence,
            "operational_recommendation": operational_disposition(action),
            "summary_recommendation": action_brief(finding, table, action)["summary"],
            "simple_action_steps": action_brief(finding, table, action)["steps"],
            "tool_to_use": action_brief(finding, table, action)["tool"],
            "plain_language_meaning": end_user_profile(finding)["meaning"],
            "decision_rule": end_user_profile(finding)["decision"],
            "validation_starter_cell": validation_starter_cell(finding, table),
            "beginner_glossary": BEGINNER_GLOSSARY,
            "benefit_category": benefit_profile(finding)[0],
            "expected_benefit": benefit_profile(finding)[1],
            "benefit_measurement": benefit_profile(finding)[2],
            "benefit_realization": benefit_profile(finding)[3],
            "prerequisites": action["prereq"],
            "implementation_steps": action["steps"], "command_or_tool": action["command"],
            "validation_specification": json.dumps(validation_spec(finding), sort_keys=True),
            "test_variants": " | ".join(validation_spec(finding)["variants"]),
            "test_queries": " | ".join(validation_spec(finding)["queries"]),
            "test_repetitions": validation_spec(finding)["repetitions"],
            "test_metrics": ", ".join(validation_spec(finding)["metrics"]),
            "test_code_template": validation_spec(finding)["code_template"],
            "validation_plan": action["validate"] + " Detailed protocol: " + json.dumps(validation_spec(finding), sort_keys=True),
            "success_criteria": validation_spec(finding)["acceptance"],
            "rollback_plan": action["rollback"], "dependencies": action["dependency"],
            "estimated_effort": action["effort"],
            "maintenance_window": "Required" if action["risk"] in ("MEDIUM", "HIGH") or action["execution"] in ("AUTO_SAFE_APPROVED", "APPROVED_MANUAL") else "Not normally required",
            "concurrency_control": "Single writer/quiesce mutations; never dual-write shortcuts." if action["execution"] != "NO_ACTION" else "No mutation.",
            "table_size_mb": float(table.get("size_mb", 0.0)),
            "table_files": int(table.get("num_files", 0)),
            "priority_score": int(table.get("priority_score", 0))
        })

for table in profiled_tables:
    rows = [x for x in implementation_plan_rows if x["table_fqn"] == table.get("table_fqn")]
    table["implementation_plan_count"] = len(rows)
    table["approval_required_count"] = sum(x["approval_status"].startswith("PENDING") or "REQUIRED" in x["approval_status"] for x in rows)
    table["auto_executable_count"] = sum(x["execution_type"] == "AUTO_SAFE_APPROVED" for x in rows)
    table["implementation_plan_summary"] = "; ".join(
        f"{x['implementation_phase']}:{x['action_title']}[{x['approval_status']}]" for x in rows
    )

phase_order = {"0_BASELINE_AND_ACCESS": 0, "1_SAFE_MAINTENANCE": 1, "2_VALIDATE_AND_IMPLEMENT": 2,
               "3_ARCHITECTURE_CHANGE": 3, "4_EXTERNAL_OR_PLATFORM_VALIDATION": 4}
implementation_plan_rows.sort(key=lambda x: (phase_order.get(x["implementation_phase"], 99),
                                              -x["priority_score"], x["table_fqn"], x["sequence"]))
print(f"Implementation plan created: {len(implementation_plan_rows)} actions across {len(profiled_tables)} tables")


---
### Step 5: Intelligent Table Maintenance & Storage Cleanup Engine (`OPTIMIZE` + `VACUUM` + `ANALYZE`)
**What this cell does:**
* When `run_mode = "audit_only"`, it **immediately returns without modifying any table**.
* When `run_mode` is `"optimize_only"`, `"vacuum_only"`, or `"full_maintenance"`, it executes the 5-Step Maintenance Sequence across flagged tables in parallel using **Spark `FAIR` scheduler pools** (`maintenance_fair_pool`):
  1. **Single-Writer Rule Guardrail:** Automatically skips Shortcut tables (`skip_maintenance_on_shortcuts = True`) so the consumer workspace never dual-writes or compacts external shortcut storage.
  2. **Step 1 — Business Data Lifecycle Purge (Optional):** Deletes expired business rows matching `data_lifecycle_purge_rules_json`.
  3. **Step 2 — Table Property Fine-Tuning (Optional):** Stamps `delta.enableDeletionVectors = true`, `delta.autoOptimize.optimizeWrite = true`, layer-appropriate `delta.parquet.vorder.enabled`, and retention intervals.
  4. **Step 3 — Intelligent `OPTIMIZE`:** Runs `OPTIMIZE` (with `Fast Optimize` and `Adaptive Target File Size`) **only on tables flagged as needing compaction**, which also **automatically purges Deletion Vectors when deleted rows exceed 5%**.
  5. **Step 4 — Safe `VACUUM`:** Physically deletes unreferenced orphan Parquet files while strictly enforcing the **`>= 168 HOURS` (7-day) retention safety floor**.
  6. **Step 5 — `ANALYZE TABLE` Statistics Refresh:** Refreshes CBO statistics on catalog-registered Silver/Gold tables, then runs a post-maintenance `DESCRIBE DETAIL` to calculate exact **`files_compacted`** and **`storage_reclaimed_mb`**.


In [ ]:
# ==============================================================================
# STEP 5: INTELLIGENT TABLE MAINTENANCE & FINE-TUNING EXECUTION ENGINE
# ==============================================================================

# Parse optional business lifecycle row-deletion rules from JSON parameter
try:
    lifecycle_purge_rules = json.loads(data_lifecycle_purge_rules_json or "{}")
except Exception:
    lifecycle_purge_rules = {}

def execute_table_maintenance(table_record: dict) -> dict:
    """
    Executes the 5-step Fabric optimization maintenance pipeline for a single Delta table
    and measures exact before/after file counts and reclaimed storage (MB).
    """
    fqn = table_record["table_fqn"]
    schema_name = table_record["schema_name"]
    table_name = table_record["table_name"]
    quoted_fqn = table_record.get("sql_ref") or f"`{schema_name}`.`{table_name}`"
    layer = table_record["layer"]

    # Initialize result dictionary with pre-maintenance baseline metrics
    result = dict(table_record)
    result["maintenance_action_taken"] = "NONE (AUDIT_ONLY)" if run_mode == "audit_only" else "SKIPPED_HEALTHY"
    result["rows_purged_lifecycle"] = 0
    result["props_updated"] = ""
    result["optimize_executed"] = False
    result["vacuum_executed"] = False
    result["analyze_executed"] = False
    result["post_num_files"] = table_record.get("num_files", 0)
    result["post_size_mb"] = table_record.get("size_mb", 0.0)
    result["files_compacted"] = 0
    result["storage_reclaimed_mb"] = 0.0
    result["maintenance_duration_s"] = 0.0
    result["maintenance_status"] = "AUDIT_COMPLETE" if run_mode == "audit_only" else "SKIPPED"
    result["error_message"] = ""

    # Guardrail 1: Do nothing in read-only audit_only mode or on non-Delta tables
    if run_mode == "audit_only" or table_record.get("format") != "delta":
        return result

    # Guardrail 2: Require explicit approval for exact-table mutation.
    if require_explicit_plan_approval and fqn.lower() not in approved_plan_table_fqns:
        result["maintenance_action_taken"] = "SKIPPED_NOT_APPROVED"
        result["maintenance_status"] = "SKIPPED_PENDING_APPROVAL"
        result["error_message"] = f"Add exact FQN {fqn!r} to approved_plan_table_fqns_json after review."
        return result

    # Guardrail 2 (Fabric optimization Single-Writer Rule):
    # Audit shortcut tables, but do NOT run OPTIMIZE/VACUUM on shortcut targets from the consumer workspace!
    if table_record.get("is_shortcut") and skip_maintenance_on_shortcuts:
        result["maintenance_action_taken"] = f"SKIPPED_SHORTCUT_SINGLE_WRITER_RULE ({table_record.get('shortcut_type')})"
        result["maintenance_status"] = "SKIPPED_SHORTCUT"
        return result

    t0 = time.time()
    actions_done = []

    try:
        # Assign this worker thread to a named FAIR scheduler pool so parallel tables share cluster cores evenly
        spark.sparkContext.setLocalProperty("spark.scheduler.pool", "maintenance_fair_pool")

        # ------------------------------------------------------------------
        # STEP 1: Business Data Lifecycle Purge (Optional Active Row Cleanup)
        # ------------------------------------------------------------------
        purge_predicate = lifecycle_purge_rules.get(fqn) or lifecycle_purge_rules.get(fqn.lower())
        if run_mode == "full_maintenance" and purge_predicate:
            spark.sparkContext.setJobDescription(f"[1/5 Lifecycle Purge] {fqn} WHERE {purge_predicate}")
            spark.sql(f"DELETE FROM {quoted_fqn} WHERE {purge_predicate}")
            hist_del = spark.sql(f"DESCRIBE HISTORY {quoted_fqn} LIMIT 1").first()
            if hist_del and hist_del["operationMetrics"]:
                result["rows_purged_lifecycle"] = int(hist_del["operationMetrics"].get("numDeletedRows", 0) or 0)
            actions_done.append(f"LIFECYCLE_PURGE({result['rows_purged_lifecycle']} rows)")

        # ------------------------------------------------------------------
        # STEP 2: Recommended Table Properties Fine-Tuning
        # ------------------------------------------------------------------
        if apply_recommended_table_props and run_mode in ("optimize_only", "full_maintenance"):
            desired_props = {}
            flags = set((table_record.get("flags") or "").split(" | "))
            if "MUTABLE_TABLE_WITHOUT_DV" in flags and not table_record.get("dv_enabled"):
                desired_props["delta.enableDeletionVectors"] = "true"
            if ("SMALL_FILE_PROBLEM" in flags or "LOG_STORE_SMALL_COMMIT_ABUSE" in flags) and table_record.get("optimize_write") != "true":
                desired_props["delta.autoOptimize.optimizeWrite"] = "true"

            # V-Order is never changed from a layer/name rule. It requires an explicit consumer A/B test.

            # Align logRetentionDuration & deletedFileRetentionDuration on Bronze/Staging tables
            if layer == "BRONZE_STAGING":
                desired_props["delta.logRetentionDuration"] = f"interval {int(staging_retention_days)} days"
                desired_props["delta.deletedFileRetentionDuration"] = f"interval {int(staging_retention_days)} days"

            if desired_props:
                prop_kv_sql = ", ".join(f"'{k}' = '{v}'" for k, v in desired_props.items())
                spark.sparkContext.setJobDescription(f"[2/5 Set Table Properties] {fqn}")
                spark.sql(f"ALTER TABLE {quoted_fqn} SET TBLPROPERTIES ({prop_kv_sql})")
                result["props_updated"] = ", ".join(f"{k}={v}" for k, v in desired_props.items())
                actions_done.append("SET_TBLPROPERTIES")

        # ------------------------------------------------------------------
        # STEP 3: Intelligent OPTIMIZE (Compaction + Auto DV Purge
        # ------------------------------------------------------------------
        # Engineer with Intent: Only run OPTIMIZE if the table was flagged by the health scanner!
        should_optimize = (
            run_mode in ("optimize_only", "full_maintenance")
            and (table_record.get("needs_optimize") or result["rows_purged_lifecycle"] > 0)
        )
        if should_optimize:
            full_kw = " FULL" if (use_optimize_full and table_record.get("clustering_cols")) else ""
            if use_optimize_full and not table_record.get("clustering_cols"):
                log_event("optimize_full_downgraded", level="WARN", table=fqn, reason="OPTIMIZE FULL requires Liquid Clustering")
            where_kw = f" WHERE {optimize_where_predicate}" if (optimize_where_predicate and table_record.get("partition_cols")) else ""
            opt_sql = f"OPTIMIZE {quoted_fqn}{full_kw}{where_kw}"
            spark.sparkContext.setJobDescription(f"[3/5 Running OPTIMIZE] {fqn}")
            spark.sql(opt_sql)
            result["optimize_executed"] = True
            actions_done.append(f"OPTIMIZE{full_kw}")

        # ------------------------------------------------------------------
        # STEP 4: Safe VACUUM (Orphan Parquet File Cleanup
        # ------------------------------------------------------------------
        should_vacuum = (
            run_mode in ("vacuum_only", "full_maintenance")
            and (table_record.get("needs_vacuum") or result["optimize_executed"] or result["rows_purged_lifecycle"] > 0)
        )
        if should_vacuum:
            effective_retain_hrs = int(vacuum_retain_hours)
            # Enforce the >= 168h (7-day) operational safety floor
            if effective_retain_hrs < 168 and not allow_unsafe_zero_hour_vacuum:
                log_event(
                    "vacuum_retention_enforced_to_168h",
                    level="WARN",
                    table=fqn,
                    requested_hours=effective_retain_hrs
                )
                effective_retain_hrs = 168
            spark.sparkContext.setJobDescription(f"[4/5 Running VACUUM ({effective_retain_hrs}h)] {fqn}")
            spark.sql(f"VACUUM {quoted_fqn} RETAIN {effective_retain_hrs} HOURS")
            result["vacuum_executed"] = True
            actions_done.append(f"VACUUM({effective_retain_hrs}h)")

        # ------------------------------------------------------------------
        # STEP 5: Refresh CBO Table Statistics (ANALYZE TABLE
        # ------------------------------------------------------------------
        should_analyze = (
            run_mode == "full_maintenance"
            and run_analyze_table_stats
            and not quoted_fqn.startswith("delta.`")
            and (table_record.get("needs_analyze") or result["optimize_executed"])
        )
        if should_analyze:
            spark.sparkContext.setJobDescription(f"[5/5 Running ANALYZE TABLE] {fqn}")
            key_cols_str = table_record.get("clustering_cols") or table_record.get("partition_cols")
            if key_cols_str:
                spark.sql(f"ANALYZE TABLE {quoted_fqn} COMPUTE STATISTICS FOR COLUMNS {key_cols_str}")
                actions_done.append(f"ANALYZE_COLS({key_cols_str})")
            else:
                spark.sql(f"ANALYZE TABLE {quoted_fqn} COMPUTE STATISTICS")
                actions_done.append("ANALYZE_TABLE")
            result["analyze_executed"] = True

        # ------------------------------------------------------------------
        # Post-Maintenance Verification: Measure Files Compacted & Storage Reclaimed
        # ------------------------------------------------------------------
        if actions_done:
            spark.sparkContext.setJobDescription(f"[Post-Verify DESCRIBE DETAIL] {fqn}")
            post_detail = spark.sql(f"DESCRIBE DETAIL {quoted_fqn}").first()
            if post_detail:
                pd_dict = post_detail.asDict()
                post_files = int(pd_dict.get("numFiles") or 0)
                post_mb = round(int(pd_dict.get("sizeInBytes") or 0) / (1024.0 * 1024.0), 2)
                result["post_num_files"] = post_files
                result["post_size_mb"] = post_mb
                result["files_compacted"] = max(0, int(table_record.get("num_files", 0)) - post_files)
                result["storage_reclaimed_mb"] = round(max(0.0, float(table_record.get("size_mb", 0.0)) - post_mb), 2)

            result["maintenance_action_taken"] = " -> ".join(actions_done)
            result["maintenance_status"] = "SUCCESS"
        else:
            result["maintenance_action_taken"] = "SKIPPED_ALREADY_HEALTHY"
            result["maintenance_status"] = "SKIPPED_HEALTHY"

    except AnalysisException as ae:
        result["maintenance_status"] = "FAILED_ANALYSIS_EXCEPTION"
        result["error_message"] = str(ae)
        log_event("table_maintenance_failed", level="ERROR", table=fqn, error_type="AnalysisException", error=str(ae))
    except Exception as ex:
        result["maintenance_status"] = "FAILED_RUNTIME_ERROR"
        result["error_message"] = str(ex)
        log_event("table_maintenance_failed", level="ERROR", table=fqn, error_type=type(ex).__name__, error=str(ex))
    finally:
        result["maintenance_duration_s"] = round(time.time() - t0, 2)
        spark.sparkContext.setJobDescription(None)
        spark.sparkContext.setLocalProperty("spark.scheduler.pool", None)

    log_event(
        "table_processed",
        table=fqn,
        health_status=result["health_status"],
        priority_score=result["priority_score"],
        action=result["maintenance_action_taken"],
        status=result["maintenance_status"],
        pre_files=result.get("num_files", 0),
        post_files=result["post_num_files"],
        files_compacted=result["files_compacted"],
        storage_reclaimed_mb=result["storage_reclaimed_mb"],
        duration_s=result["maintenance_duration_s"]
    )
    return result

# Execute Maintenance Across Flagged Tables
execution_results = []
if run_mode == "audit_only":
    execution_results = [execute_table_maintenance(t) for t in profiled_tables]
else:
    workers = max(1, min(int(max_parallel_tables), 8))
    log_event("maintenance_batch_started", run_mode=run_mode, parallel_workers=workers, total_tables=len(profiled_tables))
    with ThreadPoolExecutor(max_workers=workers) as pool:
        future_map = {pool.submit(execute_table_maintenance, t): t["table_fqn"] for t in profiled_tables}
        for fut in as_completed(future_map):
            execution_results.append(fut.result())
    execution_results.sort(key=lambda x: (x.get("priority_score", 0), x.get("size_mb", 0.0)), reverse=True)


---
### Step 6: Single-Batch Delta Audit Persistence (with Spark `Observation` API) & Interactive DataFrames
**What this cell does:**
1. **Single-Batch Delta Audit Write, it converts all `execution_results` into a single Spark DataFrame (`audit_df`) and performs **one atomic Delta write** at the end of the run.
2. **Zero-Extra-Scan Inline Metrics via `Observation` API` scan!
3. **Executive KPI Banner + Two Native Fabric Interactive DataFrames (`display()`):**
   * **Grid 1:** Per-Lakehouse Summary Rollup (`lakehouse_rollup_df`).
   * **Grid 2:** Full Table-Level Health Scorecard & Recommendations (`table_scorecard_df`) with smooth horizontal scrolling, column sorting, filtering, and CSV/Excel export.


In [ ]:
# ==============================================================================
# CELL 6: BATCHED DELTA AUDIT PERSISTENCE (WITH OBSERVATION API) & INTERACTIVE DATAFRAME REPORT
# ==============================================================================
# Implements:
# : Never append 1 row at a time to Delta; buffer all audit records and write ONCE!
# : Use Spark Observation API to capture inline metrics during the write action
#                     with ZERO extra table scans.
# - Clean KPI Banner + Native Fabric Interactive DataFrame Grids (per-Lakehouse Rollup + Table Detail).

from IPython.display import display as ipy_display, HTML

audit_schema = T.StructType([
    T.StructField("run_id", T.StringType(), False),
    T.StructField("run_timestamp_utc", T.TimestampType(), False),
    T.StructField("run_mode", T.StringType(), False),
    T.StructField("lakehouse_name", T.StringType(), True),
    T.StructField("lakehouse_id", T.StringType(), True),
    T.StructField("table_fqn", T.StringType(), False),
    T.StructField("schema_name", T.StringType(), True),
    T.StructField("table_name", T.StringType(), True),
    T.StructField("layer", T.StringType(), True),
    T.StructField("format", T.StringType(), True),
    T.StructField("is_shortcut", T.BooleanType(), True),
    T.StructField("shortcut_type", T.StringType(), True),
    T.StructField("shortcut_target", T.StringType(), True),
    T.StructField("health_status", T.StringType(), True),
    T.StructField("evidence_tier", T.StringType(), True),
    T.StructField("test_plan", T.StringType(), True),
    T.StructField("adaptive_target_file_size", T.StringType(), True),
    T.StructField("priority_score", T.IntegerType(), True),
    T.StructField("size_mb", T.DoubleType(), True),
    T.StructField("num_files", T.IntegerType(), True),
    T.StructField("avg_file_mb", T.DoubleType(), True),
    T.StructField("min_file_mb", T.DoubleType(), True),
    T.StructField("median_file_mb", T.DoubleType(), True),
    T.StructField("max_file_mb", T.DoubleType(), True),
    T.StructField("metric_status", T.StringType(), True),
    T.StructField("metric_verification_note", T.StringType(), True),
    T.StructField("consumer_num_files", T.IntegerType(), True),
    T.StructField("consumer_size_mb", T.DoubleType(), True),
    T.StructField("owner_num_files", T.IntegerType(), True),
    T.StructField("owner_size_mb", T.DoubleType(), True),
    T.StructField("partition_cols", T.StringType(), True),
    T.StructField("clustering_cols", T.StringType(), True),
    T.StructField("total_cols", T.IntegerType(), True),
    T.StructField("dv_enabled", T.BooleanType(), True),
    T.StructField("vorder_enabled", T.StringType(), True),
    T.StructField("optimize_write", T.StringType(), True),
    T.StructField("auto_compact", T.StringType(), True),
    T.StructField("column_mapping", T.StringType(), True),
    T.StructField("log_retention", T.StringType(), True),
    T.StructField("deleted_file_retention", T.StringType(), True),
    T.StructField("min_reader_version", T.IntegerType(), True),
    T.StructField("min_writer_version", T.IntegerType(), True),
    T.StructField("write_commits_7d", T.IntegerType(), True),
    T.StructField("merge_delete_commits_7d", T.IntegerType(), True),
    T.StructField("small_append_commits", T.IntegerType(), True),
    T.StructField("days_since_optimize", T.IntegerType(), True),
    T.StructField("days_since_vacuum", T.IntegerType(), True),
    T.StructField("days_since_modified", T.IntegerType(), True),
    T.StructField("needs_optimize", T.BooleanType(), True),
    T.StructField("needs_vacuum", T.BooleanType(), True),
    T.StructField("needs_analyze", T.BooleanType(), True),
    T.StructField("flags", T.StringType(), True),
    T.StructField("recommendations", T.StringType(), True),
    T.StructField("remedy_sql", T.StringType(), True),
    T.StructField("implementation_plan_count", T.IntegerType(), True),
    T.StructField("approval_required_count", T.IntegerType(), True),
    T.StructField("auto_executable_count", T.IntegerType(), True),
    T.StructField("implementation_plan_summary", T.StringType(), True),
    T.StructField("maintenance_action_taken", T.StringType(), True),
    T.StructField("maintenance_status", T.StringType(), True),
    T.StructField("post_num_files", T.IntegerType(), True),
    T.StructField("post_size_mb", T.DoubleType(), True),
    T.StructField("files_compacted", T.IntegerType(), True),
    T.StructField("storage_reclaimed_mb", T.DoubleType(), True),
    T.StructField("rows_purged_lifecycle", T.IntegerType(), True),
    T.StructField("props_updated", T.StringType(), True),
    T.StructField("scan_duration_s", T.DoubleType(), True),
    T.StructField("maintenance_duration_s", T.DoubleType(), True),
    T.StructField("error_message", T.StringType(), True)
])

audit_rows = []
for r in execution_results:
    audit_rows.append((
        RUN_ID,
        RUN_TIMESTAMP_UTC,
        run_mode,
        str(r.get("lakehouse_name", "")),
        str(r.get("lakehouse_id", "")),
        str(r.get("table_fqn", "")),
        str(r.get("schema_name", "")),
        str(r.get("table_name", "")),
        str(r.get("layer", "")),
        str(r.get("format", "")),
        bool(r.get("is_shortcut", False)),
        str(r.get("shortcut_type", "NONE")),
        str(r.get("shortcut_target", "")),
        str(r.get("health_status", "")),
        str(r.get("evidence_tier", "")),
        str(r.get("test_plan", "")),
        str(r.get("adaptive_target_file_size", "unset")), 
        int(r.get("priority_score", 0)),
        float(r.get("size_mb", 0.0)),
        int(r.get("num_files", 0)),
        float(r.get("avg_file_mb", 0.0)),
        float(r.get("min_file_mb", 0.0)),
        float(r.get("median_file_mb", 0.0)),
        float(r.get("max_file_mb", 0.0)),
        str(r.get("metric_status", "")),
        str(r.get("metric_verification_note", "")),
        int(r.get("consumer_num_files", -1)),
        float(r.get("consumer_size_mb", 0.0)),
        int(r.get("owner_num_files", -1)),
        float(r.get("owner_size_mb", 0.0)),
        str(r.get("partition_cols", "")),
        str(r.get("clustering_cols", "")),
        int(r.get("total_cols", 0)),
        bool(r.get("dv_enabled", False)),
        str(r.get("vorder_enabled", "")),
        str(r.get("optimize_write", "")),
        str(r.get("auto_compact", "")),
        str(r.get("column_mapping", "")),
        str(r.get("log_retention", "")),
        str(r.get("deleted_file_retention", "")),
        int(r.get("min_reader_version", 0)),
        int(r.get("min_writer_version", 0)),
        int(r.get("write_commits_7d", 0)),
        int(r.get("merge_delete_commits_7d", 0)),
        int(r.get("small_append_commits", 0)),
        int(r.get("days_since_optimize", -1)),
        int(r.get("days_since_vacuum", -1)),
        int(r.get("days_since_modified", -1)),
        bool(r.get("needs_optimize", False)),
        bool(r.get("needs_vacuum", False)),
        bool(r.get("needs_analyze", False)),
        str(r.get("flags", "")),
        str(r.get("recommendations", "")),
        str(r.get("remedy_sql", "")),
        int(r.get("implementation_plan_count", 0)),
        int(r.get("approval_required_count", 0)),
        int(r.get("auto_executable_count", 0)),
        str(r.get("implementation_plan_summary", "")),
        str(r.get("maintenance_action_taken", "")),
        str(r.get("maintenance_status", "")),
        int(r.get("post_num_files", 0)),
        float(r.get("post_size_mb", 0.0)),
        int(r.get("files_compacted", 0)),
        float(r.get("storage_reclaimed_mb", 0.0)),
        int(r.get("rows_purged_lifecycle", 0)),
        str(r.get("props_updated", "")),
        float(r.get("scan_duration_s", 0.0)),
        float(r.get("maintenance_duration_s", 0.0)),
        str(r.get("error_message", ""))
    ))

audit_df = spark.createDataFrame(audit_rows, schema=audit_schema)

shortcut_metric_rows = [r for r in execution_results if r.get("is_shortcut")]
shortcut_metric_counts = {
    status: sum(1 for r in shortcut_metric_rows if r.get("metric_status") == status)
    for status in sorted({r.get("metric_status", "") for r in shortcut_metric_rows})
}
print(f"Shortcut metric reconciliation: {len(shortcut_metric_rows)} shortcuts -> {json.dumps(shortcut_metric_counts, sort_keys=True)}")
for sr in sorted(shortcut_metric_rows, key=lambda x: (x.get("table_fqn") or "")):
    print(
        f"METRIC_RECONCILIATION table={sr.get('table_fqn')} status={sr.get('metric_status')} "
        f"consumer_files={sr.get('consumer_num_files')} consumer_mb={sr.get('consumer_size_mb')} "
        f"selected_files={sr.get('num_files')} selected_mb={sr.get('size_mb')} "
        f"owner_files={sr.get('owner_num_files')} owner_mb={sr.get('owner_size_mb')} "
        f"target={sr.get('shortcut_target')} note={sr.get('metric_verification_note')}"
    )

if persist_audit_to_delta and run_mode != "audit_only" and len(audit_rows) > 0:
    try:
        import notebookutils
        audit_obs = Observation(f"audit_obs_{RUN_ID}")
        observed_audit_df = audit_df.observe(
            audit_obs,
            F.count(F.lit(1)).alias("tables_logged"),
            F.round(F.sum("size_mb"), 2).alias("total_estate_mb"),
            F.sum("files_compacted").alias("total_files_compacted"),
            F.round(F.sum("storage_reclaimed_mb"), 2).alias("total_storage_reclaimed_mb")
        )
        spark.sparkContext.setJobDescription(f"Persist Batched Optimization Audit Log ({audit_table_name})")
        if notebookutils.runtime.context.get("defaultLakehouseId"):
            try:
                (observed_audit_df.write
                    .format("delta")
                    .mode("append")
                    .option("mergeSchema", "true")
                    .saveAsTable(audit_table_name))
            except AnalysisException as schema_error:
                if "SCHEMA_NOT_FOUND" not in str(schema_error) or not LH_CONTEXT.get("abfs_root"):
                    raise
                audit_tbl_short = audit_table_name.split(".")[-1]
                (observed_audit_df.write
                    .format("delta")
                    .mode("append")
                    .option("mergeSchema", "true")
                    .save(f"{LH_CONTEXT['abfs_root']}/Tables/{audit_tbl_short}"))
                log_event("audit_table_persisted_via_fallback", audit_table=audit_tbl_short, requested_table=audit_table_name)
        elif LH_CONTEXT.get("abfs_root"):
            audit_tbl_short = audit_table_name.split(".")[-1]
            (observed_audit_df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .save(f"{LH_CONTEXT['abfs_root']}/Tables/{audit_tbl_short}"))
        spark.sparkContext.setJobDescription(None)
        obs_metrics = audit_obs.get
        log_event("audit_table_persisted", audit_table=audit_table_name, **obs_metrics)
    except Exception as e:
        log_event("audit_table_persist_warning", level="WARN", audit_table=audit_table_name, error=str(e))

# ==============================================================================
# 1. RENDER TOP KPI SUMMARY BANNER
# ==============================================================================
total_mb = sum(r.get("size_mb", 0.0) for r in execution_results)
total_gb = total_mb / 1024.0
total_files_before = sum(r.get("num_files", 0) for r in execution_results)
total_files_compacted = sum(r.get("files_compacted", 0) for r in execution_results)
total_reclaimed_mb = sum(r.get("storage_reclaimed_mb", 0.0) for r in execution_results)
cnt_critical = sum(1 for r in execution_results if r.get("health_status") == "CRITICAL")
cnt_warning = sum(1 for r in execution_results if r.get("health_status") == "WARNING")
cnt_healthy = sum(1 for r in execution_results if r.get("health_status") == "HEALTHY")
cnt_opt = sum(1 for r in execution_results if r.get("needs_optimize"))
cnt_vac = sum(1 for r in execution_results if r.get("needs_vacuum"))
cnt_sc = sum(1 for r in execution_results if r.get("is_shortcut"))

kpi_banner_html = f"""
<div style="font-family:'Segoe UI', -apple-system, sans-serif; margin:8px 0 14px 0;">
  <div style="background:linear-gradient(90deg, #0f172a 0%, #1e3a8a 100%); color:#ffffff; padding:12px 18px; border-radius:8px 8px 0 0;">
    <div style="font-size:16px; font-weight:700;">⚡ Fabric Lakehouse Optimization & Maintenance Scorecard</div>
    <div style="font-size:12px; color:#cbd5e1; margin-top:2px;">
      Scope: <b>{len(WORKSPACE_LAKEHOUSES)} Workspace Lakehouse(s)</b> &nbsp;|&nbsp;
      Run ID: <code>{RUN_ID}</code> &nbsp;|&nbsp;
      Mode: <b>{run_mode.upper()}</b>
    </div>
  </div>
  <div style="display:grid; grid-template-columns:repeat(6, 1fr); gap:10px; background:#f8fafc; padding:12px; border:1px solid #cbd5e1; border-top:none; border-radius:0 0 8px 8px;">
    <div style="background:#ffffff; border:1px solid #e2e8f0; border-radius:6px; padding:10px;">
      <div style="font-size:11px; color:#64748b; text-transform:uppercase;">Tables Scanned</div>
      <div style="font-size:20px; font-weight:700; color:#0f172a;">{len(execution_results)} <span style="font-size:12px; font-weight:400; color:#64748b;">({cnt_sc} shortcuts)</span></div>
    </div>
    <div style="background:#ffffff; border:1px solid #e2e8f0; border-radius:6px; padding:10px;">
      <div style="font-size:11px; color:#64748b; text-transform:uppercase;">Active Estate Size</div>
      <div style="font-size:20px; font-weight:700; color:#0f172a;">{total_gb:,.2f} GB <span style="font-size:12px; font-weight:400; color:#64748b;">({total_files_before:,} files)</span></div>
    </div>
    <div style="background:#ffffff; border:1px solid #e2e8f0; border-radius:6px; padding:10px;">
      <div style="font-size:11px; color:#64748b; text-transform:uppercase;">Health Status</div>
      <div style="font-size:14px; font-weight:700; margin-top:4px;">
        <span style="color:#dc2626;">{cnt_critical} Crit</span> &middot;
        <span style="color:#d97706;">{cnt_warning} Warn</span> &middot;
        <span style="color:#16a34a;">{cnt_healthy} OK</span>
      </div>
    </div>
    <div style="background:#ffffff; border:1px solid #e2e8f0; border-radius:6px; padding:10px;">
      <div style="font-size:11px; color:#64748b; text-transform:uppercase;">Needs OPTIMIZE</div>
      <div style="font-size:20px; font-weight:700; color:#0284c7;">{cnt_opt} <span style="font-size:12px; font-weight:400; color:#64748b;">tables</span></div>
    </div>
    <div style="background:#ffffff; border:1px solid #e2e8f0; border-radius:6px; padding:10px;">
      <div style="font-size:11px; color:#64748b; text-transform:uppercase;">Needs VACUUM</div>
      <div style="font-size:20px; font-weight:700; color:#7c3aed;">{cnt_vac} <span style="font-size:12px; font-weight:400; color:#64748b;">tables</span></div>
    </div>
    <div style="background:#ffffff; border:1px solid #e2e8f0; border-radius:6px; padding:10px;">
      <div style="font-size:11px; color:#64748b; text-transform:uppercase;">Compacted / Reclaimed</div>
      <div style="font-size:15px; font-weight:700; color:#0f766e; margin-top:3px;">{total_files_compacted:,} files / {total_reclaimed_mb:,.1f} MB</div>
    </div>
  </div>
</div>
"""
ipy_display(HTML(kpi_banner_html))

# ==============================================================================
# 2. DISPLAY PER-LAKEHOUSE SUMMARY ROLLUP DATAFRAME (Native Fabric Grid)
# ==============================================================================
print("📊 1. Per-Lakehouse Summary Rollup (Interactive Fabric DataFrame):")
lakehouse_rollup_df = (
    audit_df.groupBy("lakehouse_name")
    .agg(
        F.count("*").alias("total_tables"),
        F.sum(F.when(F.col("is_shortcut"), 1).otherwise(0)).alias("shortcut_tables"),
        F.round(F.sum("size_mb"), 2).alias("total_size_mb"),
        F.sum("num_files").alias("total_files"),
        F.sum(F.when(F.col("health_status") == "CRITICAL", 1).otherwise(0)).alias("critical_tables"),
        F.sum(F.when(F.col("health_status") == "WARNING", 1).otherwise(0)).alias("warning_tables"),
        F.sum(F.when(F.col("health_status") == "HEALTHY", 1).otherwise(0)).alias("healthy_tables"),
        F.sum(F.when(F.col("needs_optimize"), 1).otherwise(0)).alias("needs_optimize_cnt"),
        F.sum(F.when(F.col("needs_vacuum"), 1).otherwise(0)).alias("needs_vacuum_cnt"),
        F.max("priority_score").alias("max_priority_score")
    )
    .orderBy(F.col("max_priority_score").desc(), F.col("total_size_mb").desc())
)
display(lakehouse_rollup_df)

# ==============================================================================
# 3. DISPLAY TABLE-LEVEL HEALTH & RECOMMENDATIONS DATAFRAME (Native Fabric Grid)
# ==============================================================================
print("📋 2. Table-Level Health Scorecard & Recommendations (Sort, filter, resize columns, or export to Excel/CSV):")
table_scorecard_df = (
    audit_df.select(
        F.col("lakehouse_name").alias("Lakehouse"),
        F.col("table_name").alias("Table"),
        F.col("health_status").alias("Health"),
        F.col("evidence_tier").alias("Evidence_Tier"),
        F.col("adaptive_target_file_size").alias("Adaptive_Target_Size"),
        F.col("test_plan").alias("Required_Test_Plan"),
        F.col("priority_score").alias("Score"),
        F.col("layer").alias("Layer"),
        F.when(F.col("is_shortcut"), F.concat(F.lit("Shortcut ("), F.col("shortcut_type"), F.lit(")"))).otherwise(F.lit("Native")).alias("Table_Type"),
        F.col("size_mb").alias("Size_MB"),
        F.col("num_files").alias("Files"),
        F.col("avg_file_mb").alias("Avg_File_MB"),
        F.col("metric_status").alias("File_Metric_Status"),
        F.col("consumer_num_files").alias("Consumer_Files"),
        F.col("consumer_size_mb").alias("Consumer_Size_MB"),
        F.col("owner_num_files").alias("Owner_Files"),
        F.col("owner_size_mb").alias("Owner_Size_MB"),
        F.col("metric_verification_note").alias("Metric_Verification"),
        F.col("partition_cols").alias("Partitions"),
        F.col("clustering_cols").alias("Cluster_By"),
        F.col("dv_enabled").alias("DV"),
        F.col("vorder_enabled").alias("V_Order"),
        F.col("write_commits_7d").alias("Commits_7d"),
        F.when(F.col("days_since_optimize") == 999, F.lit("Never")).otherwise(F.col("days_since_optimize").cast("string")).alias("Last_Opt_Days"),
        F.when(F.col("days_since_vacuum") == 999, F.lit("Never")).otherwise(F.col("days_since_vacuum").cast("string")).alias("Last_Vac_Days"),
        F.col("needs_optimize").alias("Needs_Opt"),
        F.col("needs_vacuum").alias("Needs_Vac"),
        F.col("flags").alias("Detected_Flags"),
        F.col("maintenance_action_taken").alias("Action_Taken"),
        F.col("recommendations").alias("Optimization_Recommendations"),
        F.col("remedy_sql").alias("Executable_Remedy_SQL"),
        F.col("shortcut_target").alias("Shortcut_Target")
    )
    .orderBy(F.col("Score").desc(), F.col("Size_MB").desc())
)
display(table_scorecard_df)


---
### Step 7: Automated Spark SQL DDL & Migration Script Generator
**What this cell does:**
* Generates ready-to-run **Spark SQL statements** for architectural table changes that require data-owner review (such as enabling **Liquid Clustering (`CLUSTER BY`)**, migrating an over-partitioned Hive table via `CREATE OR REPLACE TABLE ... CLUSTER BY`, enabling **Deletion Vectors**, fixing **V-Order** on Gold vs. Bronze tables, or configuring **`delta.dataSkippingStatsColumns`**).


In [ ]:
# ==============================================================================
# STEP 7: AUTOMATED REMEDIATION PLAYBOOK & EXECUTABLE SQL/PYSPARK SCRIPT GENERATOR
# ==============================================================================
# In `audit_only` mode (and after maintenance runs), this cell outputs:
# 1. An interactive Remediation Playbook DataFrame showing only flagged tables (`Score > 0`),
#    what was detected (`Detected_Issues`), the conceptual recommendation (`Why_And_What_Recommendation`),
#    and the exact executable Spark SQL / PySpark implementation (`Exact_Executable_Remedy_SQL`).
# 2. A ready-to-copy Spark SQL & PySpark execution runbook covering ALL custom optimization heuristics
#    (OPTIMIZE, VACUUM, Deletion Vectors, Optimize Write, Auto Compaction, V-Order,
#    Liquid Clustering with schema-inferred keys, Data Skipping Stats Columns, CBO ANALYZE,
#    and Log-Store micro-batch refactoring).

flagged_tables = [r for r in execution_results if r.get("priority_score", 0) > 0]

print(f"🛠️  1. Candidate Remediation Playbook ({len(flagged_tables)} flagged tables; validation and approval still required):")
if flagged_tables:
    remedy_playbook_df = (
        audit_df.filter(F.col("priority_score") > 0)
        .select(
            F.col("lakehouse_name").alias("Lakehouse"),
            F.col("table_name").alias("Table"),
            F.col("health_status").alias("Health"),
            F.col("evidence_tier").alias("Evidence_Tier"),
            F.col("adaptive_target_file_size").alias("Adaptive_Target_Size"),
            F.col("priority_score").alias("Score"),
            F.when(F.col("is_shortcut"), F.concat(F.lit("Shortcut ("), F.col("shortcut_type"), F.lit(")"))).otherwise(F.lit("Native")).alias("Type"),
            F.col("size_mb").alias("Size_MB"),
            F.col("num_files").alias("Files"),
            F.col("avg_file_mb").alias("Avg_File_MB"),
            F.col("flags").alias("Detected_Issues"),
            F.col("recommendations").alias("Why_And_What_Recommendation"),
            F.col("test_plan").alias("Required_Test_Plan"),
            F.col("remedy_sql").alias("Exact_Executable_Remedy_SQL")
        )
        .orderBy(F.col("Score").desc(), F.col("Size_MB").desc())
    )
    display(remedy_playbook_df)
else:
    print("✅ All scanned tables are healthy (Score = 0). No remediation scripts needed!")

# ------------------------------------------------------------------------------
# Print Copy-Pasteable Spark SQL & PySpark Implementation Runbook
# ------------------------------------------------------------------------------
print("\n" + "=" * 115)
print("📜 2. CANDIDATE SPARK SQL & PYSPARK RUNBOOK — DO NOT EXECUTE WITHOUT THE STEP 8 PLAN AND APPROVAL")
print("=" * 115)
print("-- Baseline session settings require approved Environment/workload review; no blanket SET block is auto-approved.")
print("-- NEE is intentionally omitted: enable spark.native.enabled only after representative native/fallback testing.")
print("-- Apply table commands only when the matching Step 8 action is approved and prerequisites/rollback checks are complete.\n")

for r in flagged_tables:
    fqn = r["table_fqn"]
    sql_ref = r.get("sql_ref") or fqn
    flags = r.get("flags", "")
    is_sc = r.get("is_shortcut", False)
    sc_note = f" [SHORTCUT -> {r.get('shortcut_target')} | Run in Source Workspace!]" if is_sc else " [NATIVE TABLE]"

    print(f"-- -----------------------------------------------------------------------------------------------------------")
    print(f"-- TABLE : {fqn}{sc_note}")
    print(f"-- STATUS: {r['health_status']} (Priority Score: {r['priority_score']}) | Size: {r['size_mb']:,.2f} MB | Files: {r['num_files']:,} (Avg: {r['avg_file_mb']:,.2f} MB)")
    print(f"-- ISSUES: {flags}")
    print(f"-- WHY   : {r['recommendations']}")
    print(f"-- CANDIDATE IMPLEMENTATION SQL (requires Step 8 approval):")
    for stmt in [s.strip() for s in (r.get("remedy_sql") or "").split(";") if s.strip()]:
        if stmt.startswith("--"):
            print(f"   {stmt}")
        else:
            print(f"   {stmt};")

    # If the table was flagged for LOG_STORE_SMALL_COMMIT_ABUSE, also show the PySpark buffered write fix
    if "LOG_STORE_SMALL_COMMIT_ABUSE" in flags:
        print("   -- [PYSPARK CODE FIX FOR LOG_STORE_SMALL_COMMIT_ABUSE (Fabric optimization]:")
        print("   -- Stop writing 1 row per commit in loops! Buffer rows in a list and write in a single batch:")
        print(f"   -- spark.createDataFrame(buffered_rows, schema=target_schema).write.format('delta').mode('append').save({sql_ref.replace('delta.', '')})")
    print()

# Optional Pipeline Exit Value
exit_summary = (
    f"run_id={RUN_ID} | mode={run_mode} | lakehouses={len(WORKSPACE_LAKEHOUSES)} | "
    f"tables={len(execution_results)} | flagged={len(flagged_tables)} | "
    f"compacted_files={total_files_compacted} | reclaimed_mb={total_reclaimed_mb:.2f}"
)
print(f"Notebook Exit Summary: {exit_summary}")


---
### Step 8: Prioritized Implementation Backlog

This is the operational implementation handoff. It states the operational recommendation, benefit category, expected benefit, how the benefit is measured, and when it can be realized. The test columns specify objective, variants, exact query roles, repetitions/cache protocol, required metrics, code template, acceptance criteria, and rollback. It also includes: phased and prioritized actions with owner, approval, prerequisites, commands/tools, validation, success criteria, rollback, dependencies, effort, maintenance window, and concurrency controls. `audit_only` displays it without writes. Persistence never grants approval.


Read the output in this order:

1. **Summary / meaning / decision rule**: what the finding means and when not to change anything.
2. **Simple steps / tool / validation cell**: where to start; replace every placeholder with a real production workload query.
3. **Approval / owner / risk**: who must approve and whether this notebook may execute the action.
4. **Detailed test and rollback fields**: use these only after the beginner-facing checks are clear.

Terms: `CU` means Fabric Capacity Units; a `clone` is a separate test copy; a warm-up is the first unmeasured run; a median is the middle duration from five measured runs.


In [ ]:
# ==============================================================================
# STEP 8: COMPLETE IMPLEMENTATION BACKLOG
# ==============================================================================

implementation_plan_schema = T.StructType([
    T.StructField("run_id", T.StringType(), False),
    T.StructField("run_timestamp_utc", T.TimestampType(), False),
    T.StructField("plan_id", T.StringType(), False),
    T.StructField("lakehouse_name", T.StringType(), True),
    T.StructField("lakehouse_id", T.StringType(), True),
    T.StructField("table_fqn", T.StringType(), False),
    T.StructField("table_type", T.StringType(), True),
    T.StructField("finding", T.StringType(), True),
    T.StructField("action_title", T.StringType(), True),
    T.StructField("evidence_status", T.StringType(), True),
    T.StructField("risk", T.StringType(), True),
    T.StructField("execution_type", T.StringType(), True),
    T.StructField("owner", T.StringType(), True),
    T.StructField("approval_status", T.StringType(), True),
    T.StructField("implementation_phase", T.StringType(), True),
    T.StructField("sequence", T.IntegerType(), True),
    T.StructField("operational_recommendation", T.StringType(), True),
    T.StructField("summary_recommendation", T.StringType(), True),
    T.StructField("simple_action_steps", T.StringType(), True),
    T.StructField("tool_to_use", T.StringType(), True),
    T.StructField("plain_language_meaning", T.StringType(), True),
    T.StructField("decision_rule", T.StringType(), True),
    T.StructField("validation_starter_cell", T.StringType(), True),
    T.StructField("beginner_glossary", T.StringType(), True),    T.StructField("benefit_category", T.StringType(), True),
    T.StructField("expected_benefit", T.StringType(), True),
    T.StructField("benefit_measurement", T.StringType(), True),
    T.StructField("benefit_realization", T.StringType(), True),
    T.StructField("prerequisites", T.StringType(), True),
    T.StructField("implementation_steps", T.StringType(), True),
    T.StructField("command_or_tool", T.StringType(), True),
    T.StructField("validation_specification", T.StringType(), True),
    T.StructField("test_variants", T.StringType(), True),
    T.StructField("test_queries", T.StringType(), True),
    T.StructField("test_repetitions", T.StringType(), True),
    T.StructField("test_metrics", T.StringType(), True),
    T.StructField("test_code_template", T.StringType(), True),
    T.StructField("validation_plan", T.StringType(), True),
    T.StructField("success_criteria", T.StringType(), True),
    T.StructField("rollback_plan", T.StringType(), True),
    T.StructField("dependencies", T.StringType(), True),
    T.StructField("estimated_effort", T.StringType(), True),
    T.StructField("maintenance_window", T.StringType(), True),
    T.StructField("concurrency_control", T.StringType(), True),
    T.StructField("table_size_mb", T.DoubleType(), True),
    T.StructField("table_files", T.IntegerType(), True),
    T.StructField("priority_score", T.IntegerType(), True),
])
implementation_plan_df = spark.createDataFrame(implementation_plan_rows, schema=implementation_plan_schema)
display(implementation_plan_df.select(
    "summary_recommendation", "plain_language_meaning", "decision_rule", "simple_action_steps", "tool_to_use",
    "validation_starter_cell", "table_fqn", "finding", "approval_status", "owner", "risk", "execution_type",
    "action_title", "evidence_status", "implementation_phase", "priority_score", "lakehouse_name",
    "estimated_effort", "operational_recommendation",
    "benefit_category", "expected_benefit",
    "benefit_measurement", "benefit_realization", "prerequisites", "implementation_steps",
    "command_or_tool", "test_variants", "test_queries", "test_repetitions", "test_metrics", "test_code_template",
    "validation_plan", "validation_specification", "success_criteria", "rollback_plan", "dependencies",
    "maintenance_window", "concurrency_control", "table_size_mb", "table_files"))

if persist_implementation_plan_to_delta and run_mode != "audit_only" and implementation_plan_rows:
    try:
        if notebookutils.runtime.context.get("defaultLakehouseId"):
            try:
                implementation_plan_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(implementation_plan_table_name)
            except AnalysisException as schema_error:
                if "SCHEMA_NOT_FOUND" not in str(schema_error) or not LH_CONTEXT.get("abfs_root"):
                    raise
                plan_table_short = implementation_plan_table_name.split(".")[-1]
                implementation_plan_df.write.format("delta").mode("append").option("mergeSchema", "true").save(
                    f"{LH_CONTEXT['abfs_root']}/Tables/{plan_table_short}")
                log_event("implementation_plan_persisted_via_fallback", table=plan_table_short, requested_table=implementation_plan_table_name)
        elif LH_CONTEXT.get("abfs_root"):
            implementation_plan_df.write.format("delta").mode("append").option("mergeSchema", "true").save(
                f"{LH_CONTEXT['abfs_root']}/Tables/{implementation_plan_table_name.split('.')[-1]}")
        log_event("implementation_plan_persisted", table=implementation_plan_table_name, actions=len(implementation_plan_rows))
    except Exception as exc:
        log_event("implementation_plan_persist_failed", level="WARN", table=implementation_plan_table_name, error=str(exc))

statuses = {s: sum(x["approval_status"] == s for x in implementation_plan_rows)
            for s in {x["approval_status"] for x in implementation_plan_rows}}
print("Approval summary:", json.dumps(statuses, sort_keys=True))
print("Mutation modes skip tables without exact FQN approval; persistence does not grant approval.")
